# BDB27 — Combine fatigue slope vs. in-game intensity decay — pipeline runner

**Provenance / status banner**

- Repo `conormalone/BDB27`; binding spec `TASK.md` (§6); all thresholds/paths in `config.yaml`.
  This notebook reproduces the **combine-side** results committed at HEAD `f45e3af`
  (Phases **1 → 2 → 3 → 3b → 3d**) — run top-to-bottom, no CLI, no Internet.
- **The combine input data are COMPLETE.** Therefore every combine-side number produced here is
  **REAL** in **both** run modes. The `MODE` switch only governs the *provenance stamp* (and the
  not-run game side): `full` ⇒ `FULL`, `sample` ⇒ `UNVALIDATED SAMPLE OUTPUT` (developer dry-run
  only). **Never** read a `sample`-stamped combine number as unvalidated — the combine inputs are
  the full, real dataset; the stamp is bookkeeping.
- **Game side (Phases 4/5) is NOT implemented this cycle** and REQUIRES the full Kaggle game-tracking
  files. Those sections below are clearly-marked placeholders; no results are faked.

**Self-contained.** All stage code (`src/*.py`) and `config.yaml` are embedded in this notebook as
`gzip`+`base64` payloads with a `sha256` manifest (next cell) — no `git clone`, no network at run time.
At runtime they are written under the work directory and imported in-process; everything still comes
from the embedded `config.yaml` (no new scientific constants in the notebook).

## How to run

- **Kaggle:** attach the 2027 BDB dataset. `DATA_DIR` auto-detects a `/kaggle/input` directory that
  contains `combine_tracking.csv`; outputs go to `/kaggle/working/bdb27_run/`. Just **Run All**.
- **Local / headless:** `pip install -r requirements.txt` (or `source ~/.venvs/bdb27/bin/activate`),
  then run all cells. Set `BDB27_WORK_DIR` to redirect the working directory off a small root disk, e.g.

  ```bash
  BDB27_WORK_DIR=/mnt/project_data/bdb27_run \
  jupyter nbconvert --to notebook --execute notebooks/run_pipeline.ipynb \
      --output /mnt/project_data/bdb27_run/run_pipeline.executed.ipynb \
      --ExecutePreprocessor.timeout=2400
  ```

A CLI fallback (identical logic, per-stage scripts) is documented in `README_RUN.md`.

## Limits (see `notes/rules.md`)

- Official: **≤ 2,000 words**, **fewer than 10 tables or figures**.
- Internal (stricter, `TASK.md`): **≤ 5 figures**. This notebook renders **3 figures** and **5 compact tables**.


In [ ]:
# ===================== Cell 1 — user params + dependency guard =====================
# User-editable parameters:
MODE = "full"        # "full" (default) | "sample" (developer dry-run only; stamp only)
# ==================================================================================

import os
import sys
import importlib
import pathlib
import subprocess

# ---- dependency guard: install duckdb/polars ONLY if missing (no-op when present) ----
def _ensure_deps():
    missing = []
    for mod in ("duckdb", "polars"):
        try:
            importlib.import_module(mod)
        except Exception:
            missing.append(mod)
    if missing:
        print("Installing missing runtime deps:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    else:
        print("duckdb + polars already present — no install needed.")

_ensure_deps()
assert MODE in ("full", "sample"), "MODE must be 'full' or 'sample'"

# ---- DATA_DIR autodetect: /kaggle/input/*/combine_tracking.csv, else <repo>/data/raw ----
def _find_data_dir():
    if os.environ.get("BDB27_DATA_DIR"):
        return pathlib.Path(os.environ["BDB27_DATA_DIR"])
    kag = pathlib.Path("/kaggle/input")
    if kag.is_dir():
        hits = sorted(kag.rglob("combine_tracking.csv"))
        if hits:
            return hits[0].parent
    # walk up from cwd and from the notebook dir looking for <root>/data/raw/combine_tracking.csv
    starts = [pathlib.Path.cwd()]
    here = pathlib.Path.cwd()
    for _ in range(4):
        starts.append(here)
        here = here.parent
    for start in starts:
        for cand in (start / "data" / "raw", start):
            if (cand / "combine_tracking.csv").exists():
                return cand
    return pathlib.Path.cwd() / "data" / "raw"

DATA_DIR = str(_find_data_dir())

# ---- WORK_DIR: env override, else /kaggle/working, else <cwd>/bdb27_run ----
if os.environ.get("BDB27_WORK_DIR"):
    WORK_DIR = os.environ["BDB27_WORK_DIR"]
elif pathlib.Path("/kaggle/working").is_dir():
    WORK_DIR = "/kaggle/working/bdb27_run"
else:
    WORK_DIR = str(pathlib.Path.cwd() / "bdb27_run")

print("MODE     =", MODE)
print("DATA_DIR =", DATA_DIR)
print("WORK_DIR =", WORK_DIR)
assert (pathlib.Path(DATA_DIR) / "combine_tracking.csv").exists(), (
    f"combine_tracking.csv not found under DATA_DIR={DATA_DIR!r}; "
    "attach the 2027 dataset (Kaggle) or check data/raw (local).")
print("combine_tracking.csv found OK.")


In [ ]:
# ===================== Cell 1b — embedded stage sources + manifest =====================
# Verbatim `src/*.py` + `config.yaml` (gzip+base64, sha256-verified). Generated by
# tools/build_notebook.py; do not edit by hand — regenerate instead.
_EMBEDDED = {
"common.py": {
"sha256": "d2560e411d803c83e078d5eb9e59a5b25bfc91bf02c1a81d07f269f11981ef59",
"n_bytes": 7327,
"gz_b64": "H4sIAAAAAAAC/7VZa2/byBX9rl9xwcAQmVBKNi2whQIvqthyNljZ0lpyiq1h0JQ4srjmQ+GQdlSv/3vPnQdJyU7RFmt/ETmcue/HuWPHcWbrsBARVWWcxGUsJK3ygsq1oI/HH9//SJt4I5I4E/1OZ1rkd3Ek5ICWebaKbyjJwyjObnzahOWaCiHzBGTyzCdZFtWyrJhwkt/cqE0hbUR42zsfntLbzn2YJL0yTgUVVdbjPaLwKRKlKNI4i2UZL0kKYciDscjCbCmYMJYkqGURhZ1UpHmx7clwJeho9qX30zQsvlaiZAnvRAFq5B5Xy9vjj3xShKB940GVYyHjmwy8EyjszoezX/pp5A06PToTOKc0o1WVJExVUpyVOZTMolB+sKQJS5YbzJURnoot3ccwhWbZB7nRt00SL2MWKKnSDDolYsk2+gB6RZhKfIARVkWeGqv2t2GakJvllIY3sEJWpQtRSE9Ru2MWsgxvBIWbjcgiSWxJUpZ8owxMbGDIllflpirlW9g3gH37S3nX7ziO0+koZkGwqthBQUBxusmhT5hleRmybLLTMWs4ZB+NH+1rLu0T+70qlsK+sywdq1ApvpVJvLA8zEoaZlCh0Ls4eFpbpnjVH8rtBgzt+jDb+vQZHg0XifBpJmBvREQtagSbRwv7BqttthRKyjZ2ie3a6ZyPppPgfDKZDxQnOlQ/LswRJzCG11dhfCdcrw8Hiaw0P53j0cnwYjwPjiZnJ58/1adrevSWnJYHYedgPPn0aXQenA1PR9jpLKLF+x+x3nlFvT/vj16B3pFi/KcTjsRKpUKgFXPZVQPOJPoDAdBnE4zjW4G3szwT0JF/POr9RFG8LC+x0We/XQ06hD8E35jzaifOwSKsklJyyHLVKcQm7xV5XpptXp9jlo8vVzfBpuUyfvYoXunqIxIpaNdH6pTKxxyp4trzPjmF4xOCJ+fycuhU5ar3N8fjaFmttaiGH1ixlH2uLwEbwl2tPSvMpWPMoqg6V9gMhWs2el8hkGQZb4ff2Zw2vLAy2LOST7diq8wL6b6VRWhNvW9cVr826bkmyOWV7aATD9J1+VV2ry5B9IrcfMOZjVqxpd9z1PNIW+ZaMbpurLwIpdgLa6WrouZoam3N1H44QdGhWGphlTfUp7f6i9H+RpSBLvZugkKbDLi0gp0pLv3PZycTpaJdGKvNLWUV05BbwlJI2btHQ+JIgQ0Eacq1Kvq1RR3cNT23nZraUVAB5c+SWKPWJ6i6TTSsW3RmqpX8rPe4XrOnL0V5khdpWJasodnerDgHbiiXXCA9SQfaBFlo3lLog6roScdrSBp5wiiy7ExgmQ9gOGYqmtZOzOkdxvBNC30+8tI8EirclPXxu29yzs1WH07ChUgUVAi5g6vz5HYvzr4Mx5+Ph/PRMc2Gp9PxiCYX8+nFvPtH9+RiPO42gWakNClz6TAFxBf/XCHcalbOlWd0YDwQcHvelmvY1eV3FUBKZo67WugZPpke4DJSkGWEFuPR+dknDXAapJHWEmWbfoHNedpnyoq8iQ3T6NTHjpb+ycaXKOtT28/fNG3+RYo8A4cAcCRIF66y5grVrqzNOdXZpvEFKhiSDmmL2CMZ/wvpn9Fp/JHccZxV3xAOQRp+K7A9ZuREv8Qfn3jdYgZOyaLisHfrpfOL2fDTKJiNxifoxjWxt/TDu/d/7b+Dof++ByNYAzZNFCho9HyEq0+mutbR7tPr10h7xrTY5dUKH2kGZDgMNMAKLfRi7KngF0d/fv89uGWVLt+heDCF/kYUK3SNKivrwiGKQtd5IASzvdg2dWcbiyTSG78txaYEouQf1HLuV1gbEL3iXNcgtBC9IoxRd2VOqzBOgPAkAcDQXSxjAKeaLtiC48p5AMoSLuh4/SDgWhQEjwN6wMKjU29WJNXbKlY9pJGP4zKQz+tHPahe79QWC4yB2EnGKdofviHl7wajz5L6pJxkC8EeJduQn/hcfWj5Xb03vu808g90wOslxT9d7KzteEkvmbjZ5bnbrf3O09I01IHDm0zoKNhTqYJN7lqEEZrWfRGjY2TYBpMXsqRKCq/d2SwUetKoTddBKXWMgZyrnUMW16a3UVy4+kUezotKMOzA6BXkt+rVMx5HALKDLx12McyZbgCfHGVXflCFG7/akvzUciC/wnY5Iy5HWcxIo55BVf+iIj881uuBMfQHcvoMVlyE6e3j4cPdo6OK961Pd1xZ1Nl+XIpUuqZnst1EYIx4aFu61lspJ03aNbjQfoeAIaTMxD3Pu4fOf4MR78GDM/0YQfAPZl0AIfraZpxM8lCbr2nowBltGRtSilxffdOfWsDCrCNe3J39DztvKsAaH+mi1YctV/wA8PFb7yDtHUTzg58HB6eDg5nj+U8JaL8OTGo+/a7cPdAp+/SrCYIBYrvKItem9F+e49QOE3vA5J5PPzx3QkfSQJWEp191eA3qENrd8li/vUyv/vx2MuDbAg3Q9b2A27qf8M0NBIDJYmtuCLwXaebojplYlm65Br9Iaoz9zIimBua+lmS6PdKn0FjqUjVBeqDDmQuUZb1BJ09IC3YZ9NGMSNX9ZlzDvkPLxIpUY20jG0MEztGmQpqjSFaxrEqB1J+eDz+dDu2JwwdoY1XzHp3dMSvPTItAUqqk/u6cFeSFancN6P3OVMUu1XMvxL5unbxm4dkAhZ63+AIB85bPpRrShiXdhUklGniJM1wuuEirvc4Vwx+3RXJHMm8fMJmxkefX8J4rldqk9S1zqKuCzrXWH3zfw6YzFssAdtJXGXopQvU3dNrL+vIKkWRvXdiYT3qd6hY5INwAkZEnWD8JMQf6rXllsDfYPUciq5KEB5DvMXOdsyF3E65eTx13ZG7ntONad3R3cR3JR5Ppb+TadPQh0z3pTOW7QSY0u403ku4ZUV23bHKtGyS7PWE8fK3UVc3ymtzlWixvN+hXpb5lZELXRptrQElQ3CThUqzzhIOmzG9Fpu49Zr+O6QzT0Qd776pvHTmOWFc2qiKGGV5EPSBN9PoqC++A7/gyTMcZquz9Ol6uKa0AF84ACBYC4QTEiJ6VcPcIE3t92rfW2nf6/4gNkBHtw7bBKoDMaa2jod37ktY8vzfjxtkqd52NvcANYU5ax4jDAwYVbT5NdrQypLWhPdCbIK9lG+yCWjpBJp7lmNqRySNuL6g4aSwl3zzqgYSdAURsCNmKg5Rg+OK8diwnkyT67oMD1GCXrvMALN1VyGXJsMVs9KxPFR3d8w8xDQmnlt9kQi2y2vzmkKmbb4eXDobDFjen+yAfuxonqfHLbPR425UmLb9ycjZQYuXojMDINTqa0wPr9kgn55NT4jLLagdhVeZut7ZC16cHlubR88hpEZpPqPvQcsVjl9yTyfnpcE7T4fmvF6O5j/Q7nZ6PZrPPkzP652x+7GkCdUjtx8hOfJjbd3bPgeTsV9FhvayL6POx0u4qsMBOeW0Hj7kw+WpuO6fqmvH5S5G6oigz2QLsIXgTm2vKEddM6np/CF45O8e6D7zrseu9zBXxLGWh1iLBjCZfBHdoXAs3tCzn85CDxmEv7i93W/FVGyv/x/bydJBSeBulhmJDm/KVGqmYhaqrSNza5v/H5MOS860fPrj8vHNRqD5ynWvkt7ef+yXmC5dnW1uW6t8sSMwVWoCZUzhERLopt1SyHig2KhKcvQFMCdKwU3ffLe6q7tTCXr676gNMNINRPe+YWef+5Wed740zrG/BtUnFxnMTEE86hdf5N1k0qkyfHAAA"
},
"01_audit.py": {
"sha256": "976d082955111268aacf01c0e552a56b3e34761f27da112fc550457df175bdf4",
"n_bytes": 50375,
"gz_b64": "H4sIAAAAAAAC/81923bbxrLgu76iD7IyBGySIiU7Fzn0WbIlJ5rIkkeSk5Oj6EAgAZKISIIBQEkMh1nzuPf7+Zd5nw+Yj9hfMnXpbjQuvNixZx+vlYgAuquru6urq+vWlmW9G3pJINrCm/lhKuxBtDuJGoNIDLw0cEQ/ikU6DMSro1d7X4teNO6Gk6DR99JwMAtEks78eXNn53CSPARxQiWT4D6YSGi/z4IkDaNJImyvMQBocTQWV4eXPzbHvngI06EI7kM/mPSCuvAmvgjGYZoIb2caJSHWawziaDYVu8KPw9EImh2Ho7mIA8BjHEx8D8s0xaU3no6CxjjyAxHN0ukMYcSIHXwI/J3b2/dnPx2enhwdXh0ficvDt+9Oj8X5+6t3769ub18g0nMxniWpmADqsegGIpykQTyNgzTwhZdAg8lslCbQ0YvZ5GBHwL/pPB1GE5HEvd1W26XeNqdz0WAsEsJoi4L92Wgk+N8XMOLjoJHAeAgbRj6ewbANA+hHNBnNd3bO4f/iaNa7O3qF4xqHQSIixPedF8NzSl2eJYCxmrNh4N3PxSjsp+Fk8EIkYw8a477spF53FPAwDWHkR4Gcjyk8eElTnEWM2+vLn0SYCBqYUeT5UA4GJ1LldizL2tmhaXXd/iydxYHrinA8jWJAaDKJUpqjZGdHvYsHUy9OAvX8WxJN1O9knjCoqZcOR2FXwXkHj/whnU+hK+r94WSu4fowMH5XPTF2OHVTf2cH4DYRZDOcJEGc2q06kEZsI1gbsA5HgLPThIGJRveB7UDZOJikjsNtIq3BBErINk7UJPrdOxDHz1p7NMUXx+/O3Yvz86s6PfaiySTopfIhuXexbX4aBKk7igaDIOZnHFEXyvfDAb+YBt6dG3tjd9yVL36Xf2OY64mHK4WeJbb8kASB7+IUAa3B+PDLNBzDW1gDA1kqjQATIpX6jrOz8/r87auTs2P36uLw9Y8nZ9+7r89PL0VHXFNhy4+9furOAy+26sLCNZ26oY+/J/2R/AXvwnTuwqwE+IgtWnVVHRas/sJPE29MT16aBuNpij8f8X9z/F9CX6hwmPAfaHnnRuN5cXz5/vTqci2aGWqSU7mKk5jvhkE4GKbmmwf5hmHignCT8A9GNh67o2AySIf49ADD6yZAXtTfYOLOoeR0FBI0WHcp9QVmIg173kgB7MY4z7/NxlOqNoyDAGed4CdDqOUmw1majuhFF9jh0I2DaaKqTwaJiysiAKBhMnaTXhRTUfzQi0ajYAAdjSNYA9jV/Pd+OPFG6hUM5w4wPvfy/aujkwsYRYsYEizhnS+AJX2yf+ILgDcMRlPYFD45ZD/oC3fkdQPouG/7/QNY5M0jL/XewMKBjQSXygGucEc0Xua+MesGlnWJOwMxNdhNogcR9YFZyT2LOCMzQ9yMsoUnqM0mcjwE4/dh/Px+sxdN57YjXxksxspqWoyUI1cucMkJlIVBp648xGEa2L3+4ED4YS+9BszryNtu6gJXDHWlLtb0sy6Yp1B/katxP2ErdGERAZaKtUEbgJfcIgGpjBQcs0ZzfAf/t5kNJp2reAatBY9hkrrRHT1yceRrAF21s0vo0hdzemTfm8B+gBnaxAxhD/GDx84bb5RIWNwDGL5+ZFsPcZQG4kuQGr70cX4SB0cQOfiEeg7rEQA7ufHEz2pEE1h7XmzDEgNO//soIwYYVR6be8AbPjeDx6A3g9GHUk6zH6Q92KsDOwf4DN6IsA9VYB+khwCwFvfXrZvPsmxsz0HejfvcrohiH3Z6+NkPg5H/edYSkb3r8XBdSYqqIEcawfwrHkw/SHqF8exbR8eXry9OXh2Ly+PT49dX4ol4c3H+ViyulpbT9PtyjLlbuJAAxvU1MOTRbDzRO4V8pG3k5gZJCNu3LZQAYz+xYBtj0UmPkzcA5gqyYQrYRyDg3MrN5hZIrhcHKOrS0r7FzeqWfocTYcPGURfZLuX8K68HhOoCbQCCfbXufz65+kEMxOGlsHek6Kb6yNuPCQjEWm6/LmBGbWzUwaopMIix92i8aGtgapzEzz8cXxwLY5MFNGrvTg9/Ob6o6dLfX5y/fyde/SLadbFXF/v0wQG6/3AMASldXv4beQNbfnfE+U/HF8J+d3hxdXJ1cn6GjZbhifOLIygG39IW9QzE6HvXq4Scth1++AuQiwM34AHYMTrdi2aT1H7iiDcnp1fYEA8sIyZOLsXZ+ZU4e396ylC9ME5y6G5dXxyeHanhFC+FLEFQiS4/AdTvOibU+zAasYD9KRGmt2lLvOyoIc664NKaCPzP0d53ItceHm5G3nRHT22itu9scRbYjl6wzGKa4SjqAZ/WjEPtWzDDUBMOMlzj2qJX1g1/5472Yw+5GhxAJ75tFKWvUBS4M5OKeObg/sBQaW/AXUJxJiU546ls7AFb2mvt7Yto6oEgLu4TfHy2u/dcHVaFnT5EApoEyQlltoRRglEpsCFJ25n8WzEjRyeXVydnUEohQeM6cdVjnmqQQeUKBo90jM0V4uHw7gf268PLY2RRZ1kXT09+PBa1L3/91aX/agL2gMN38OrXX2viCou2my1xfAoVW/jj7AgY1T61haPtQtddo+s7OW6YMbqMEbRzFBEUd3UeNUkLqzaP9A89tHJMoWdXtjeZu/feaBZkPPqnw4vXPxxe5MamwNZdeCxydhces81PtlnAlJFYSbXqQIenD9oyofpCnrey89SyGSbJrJsEqb3o5/fSG1IK9IHk5Za7lKTuoniFx6mF1RsGvTvrAM4xWAIOFEEC80oHG3wBX4pA8ZRGe7P5iTfrZYHfF5q/MVt/qk5z+M/AQ/bLxX66vEINZNTnP9UwKFTwbE+lnWV9I9hsrW8NO6uyTQMZi966AYPZGLVvnG2ao/9JHr2pW9kU5Zqv5ngK6GbOtwJNwoxJa+KF94E7idz0jxx6a9Gy/t0SkyhFGsLH9I9rSy5CRAm1h9bTNSXWYQXrqgKPfPMIjZY3wFoHynvcFhSU1KB4PeAiQeUjcLID3QIvkqY3nQYwLUZ7iu26vLPkFqqhF+ksguta9li7WVolYaw04H01clRb/oaqL8SE3hh7CL2tYt9UsOoDorDMnbEWGiHuQoIo0I8MVYs3d+YDB0UxTX41lvOBsY+Xyhnr6kCsWm9GLZrbMGHKhSrdKBr9ZYo04OMUI8Y01/l2Nc/XHFk2b36TsJaf50TadZTQlogpCF3TkTfHP/Afied10Z0LpWkTpLP/PFofxeqCx2lcrTHxRqGX0DJD7VZq0aEVnjLVz/84pXWGB3kliRJtksqcdbEz4HMsQuGRcjZBu8hRe8/Ruh961cFD8rXUoN00B7Dtal5MlV0sltOeMtEDy2QAHeNTtuDlmuhbBSFuQX1bNrWMZu2sK26r8uZZT71TZzrHkjoTVgJ080qAiw3KgErd2kZqEVe/vDuuIJkmizmv9mDpPx7QdEQg6dzeMoDEHQR7igKS21uRzMbjAL/b5bOig9/JRsJ6d4QP+1US0dx7YjwbpWEDhKUGo5Spq4UdNAdNgHr548npqXt0Af+/dI9e3d7CXpciVYx8gnn8b6+Pj48M9FCX4BIgaPwf/+s/sZnwMfAbg9ibzEZeDMd4eJOwGlEck/ZRDg7NHYGFb9EDqpom8McTajobfO7ngi+koQ37J01k9B6r3QVA0Ekw9WIvDUZzlrMmSM4xCL7paE5aOjhnKMQZZQbm8kBBX+VknEVpcAB9zMYH+uYNgLGieTAhDan6inBpWuSg8nh7sos1nonb2wnOIZQj01ZINsMQFl+UeiMRgfAa36PRTRGQVNCkQy81JqkprobSvBbEDUNF0I0D786PHiZiBIw6QcZ7ezvUVON259zdZi+5v71t5o4PQI2FNQ3nQkmhiRv1XbIKBgl8UOXdUZggI8BF3vwtgqNA36otpsuaRXQ2xfahmCPPgmfPmLS9XjrzRjAV7+EktI7tAB3ycCmCoHmfjVFhncoR+MKszbwKlV63sByCvgd0np3+mG7U424yilJujw2BRDpeIqH6oTeYREka9pKmeCNNir1ZjEph2U52WsRvbFlETNGkSwYQmFa1jiXUW8Depj6BBOHcAgyvN8xOkEj9k0GC9WCjJHJFRm2v0VvJre8LJDZkCTh2PkxLOOllXWdLKaJiEAtSKhO86JGtUvjRrDsKGjQmzgsJF2tNefWGaFNEBX/AqD4wBQEmQASl3alOGxBb6Cp0iLzkKtV0aVN1OG1mVJ890QAUhbgFtL8UK//RyR/HHeeuVLmweaR6hyhURprZXDkjjFzl4L6s5EylmhMqfZyiczqlIYQHkBOq9huCeLCSw9iF5eOs15nSRBQHICNqgqB7vGKknji5STE2qkzXxcSR67Tu8VrVbtHySTpNYG084hdKrVwqdnIGIoPiaVI8z6lRp00Ew9CoCu8WOAPNFWNTIAwoyFg6VdRZ3ke3UjICTBxq1Fa2nSKsQdDWzP9Doe1VQNv7aGj7FdD2q6FlOjYFACZ9z6le01CsGgoqoXR9se4fQIHC1VCQsreGQktsAxwmbGctnJWCSaYRhFX/38+BZKfiHP4nyQrYhiaxnRLPoDeZ6rBeQFegxapAzDn5gAy+JaOhNmXh9n6gfW/WbZQZw2nQZ+ZM5IEUB2zHD4ztXVVudD307uHScndsiu9nXuzDtjTwwgkA8EQSjnTDIKbBrpygl5Q6eXz4mWXVOQU19YQLa9JzVletS12piNYbgLUjNv/rW+uMYYAjDHsr49zYiRxyfv/ayk839B5p0iENljQosw6rlaleaK/PQevobtcNFswoyjMUDz0I+mMv7Q0PVkz3IoO6FP/SySQV+Vk1s7TKbgOf4Yjfc9heyschtq3ce3BA8XpxlCSKaf3rZ7RA97azQKez6Si4LrhC5J7ylaSRGs5gd5Wm3G0MpSx9FSy4H2awrd7K47utW4+jB3cyG3eD2F5jii0ZSePJXUGyKO/sa3fvJ84aTp0wd1m1f0HjFVuX3LVwRirqJanvo+HGG08L9bE9v1wNpwULrtvewklFPdggN9WDbRHrZfsOTFiVIUr3R6zYPPBjUtg/FEnKTYT9BeGsXj6G4tvQhxO3m4FP5DEUv1WfQ311DkWfTYacGUD/qWsBDb6GyoVET92PpVMt86s181slkp4+MnnGBo5F+cmri27lh25JSM8sm16TzNJd/MP2S2m9RNsl0bHbD+MkLaww2IVJOElEF8UTLxNPuuonGcJNfL7LobdqjXp1jXW2POkQIRn0ioUo8XTIZKPMrfLlhjVYrApL0Ow19fi3/EzlpKy9vJJF2t7NdaDIMSdM8RYEB26rj0o0SyrusCzu2ChHYSfo6AKl/myhCPBnW9iTCNVu4WTgNCWoqyEIQUPUJWqlEO/Jwn4FItuDl4ghCFFw7Ef35laz9VxWTYeoRe6PIi+1y3ITWxL6o3DqpqoFWIBUn3uB6j8yrhKLih6uLXPc0X7aFg1R8YH97l1i+qRDwlFrhmkQo33Adm4M9yTYpUNqhYwC3CQq/JC90ZMjXmJP1MA+DMPeUM6DVkRhL9jnCX+RkzWgnVmlVqOSyYN98pYo9UV/h1nDcehzr/sSrYO8ECVbV+auvrVAkDVJ/bWb5Z/miy68sP/x978tmGz7uFcsHaVzmY3HKMF0TPNS5aShstnwn7CQ7WM/H8Ik4LIJEAjafbidbGiVCZSHXZtADViMqDlVLotUasUCVPOrUVUNhrZ5qRdGmUmUIl4WkX9bdF4KWi7oc5lGvWgk2LJFBI+4Kz9/lvOQ1TgvONxhHPgh7DAIAjGRsRVl6dx6c35x9Yv7y+HFkXt0ePmD8EYP3hwdZYNe4AccBOINwhEqzPgI94JCGKSSEO1r94Fc31G/qoXLH6AJ9/KH91dXp8foJHP1w8Xxsfv6/OyYFfJCEr09CbxYPG/tPm85TUvZwAxxmbbdujJXS4r4PBI0nGnQ0TXEIUb9I7kMs08E9jr0ujwk0B39mQIMHnfnn1Gq9j9eqq6Wo9NoVMkUry3VLRfLKGeqKjGjX7mDqxNQvVLg0F6K0EpdPNbFvNKl8LFSNtaHq0w6VrLL9LESzvyD4cw/woETlXmwCj5+NKoUkTBAjvjof+RXyWRcCT35PU5t+xE4+BTG+onQP58Ke44/5/RW/nScqtN9QUv2mPfB3AJDigCbguzgb9SwftQITNBrYFzUyfZLOmhg/5/sNLVOZ79OR+h1E1sznoYxNA7vTvkDTeABwG7i+mG/7879Cpi/zzwg2BFFqKRroKOk81xLhbR7fErQ32awp98+3wAYjZsaVj0P6rlTMapYoQIO9yLDapdI8uSNbRCcaKmBzbod425Z0ndm54isOgg9LRb+V/YeWs2VXyA7XVZqyUpnEmcDLaEZjNjzRqfOz4WyiXjZGTRHr3TaQMErQ9tQAAPfZP5htGzwkYL+N+PYk6LG9+PUvRliUIz0i8boKuUiRq7Sl81aR/IZJqkegGCgZk5sJaAHIj9zljE48EFGsQW+llN5k85K8WqAlg3kybCcGoEtBfkVVSe4WgpAfXL4hNc3TRTqHQ261CkTlrFaUF0yQNH1ugjWLGWA33fqmzTFmyCB2M6QbjZg+oWU2EYcfEtOjorw/+yYFKf8E6IRzBqJdCCHk1g6go0Q51FNvTGZErZy5FKI4KltxcwUPn3Hp8yKc8CldIoBIeCW/Zoa0sKP0jeImhy1JTmdoKEBYPvPGsBy2+iUQ8vhRZVY/k2r0W61vqRoObWEkCcDQi+ff+k0CXwae78FvTQi+ZyisMdRRC16j6LdEj/8AYJoMK2C327uibn/Anot0mAUYLxt4pCrh0CfmGetxlz6sbCPnrDjTqvZ/tYhuzyWadDJBaehArwXp0EfTrzCxmlhaLoffz5rQdvQ2kUuuPwgm+uC1F7RAHT/BfQ/8FI5/hiC7N+HSRTPiY4qjygYISfX/Oc5mIB8SpYIIIvXbPRFgf/szalS6rPVPOeLJWyK/uaQesDc+YwnlIBPKJXufIiAPLlUhzuWXdBgreKyoaMJVoeTCf6B80k8m7gYY+9ijL11k2kecsGHOF5ZJD6NHKpDLn88effu+AhVIF8mPDAYbQnb1snZ9+INBrdcvD+zGGWn6MxnomlfL3K0YwFZpTM831tlcPmSfpB6IW4EVmFGGy83zGcc/D4LY3k6p7ldZXezqPOwjHt3HHg4CuAArxke88OJmcCgaaC5vFF6Jk1AB9I5TWGHqh2PgvUADDYmVGPkFCgdE3QH2IEvde+CudKL56eVplSBkPomKE+Iqxr0AFUYzo0ilDBBoykualvVoNwAhlbLgHStfus2kJfSF/Kbz0XZylB7IAjKM0BOiU1ybUhm/X74aFtNVQROSbRtkFMSjkmhGTo/A1vxVfy+vVBtLx1LbWXqFerV2rypFWrVdLXr1s2yJv1HMQx8+vtK7HWMvEqCkWG98nxPM+3TsVaeibQBWKq/+Yg8SNUZWb42z4HmkSpurvBqMf1QKg9bJOvLqhUa8ryYrv0IzJC1ChmbLEO0/thZhSgQNgxvo1S9poVKYVi1hDJMYMRDXSxFLE6P31xJLwge8KIzRFztC7EqfkryqmqZ97NsTSD0jMMEFuFgEiSw9NAmjisQXeSRQWCv0dARxX44QYfTz7gN9ddsQ3mH6A9UmEUPyYFAw9Z14Tur2GUINaDh+64cDZuSEKiNDyP6pfYuGilY8MyaO5QPM3Y1QUfENZ4YT5TnBUJVvgVKy99j0wi0kdsbJmWgtTVA5ZK2Fr2lRXQONF7L64HyES2pFIbpr476hhc9zGHhytgIdOOYrDsDQNEJ7Dmq6IR9QtZWgOLutJfqsw2It7D6nghdG5MaKB3/JDspLKW11pwyzSj1VgR0UxeVaVac1bUlm4XKF1llM/eJbNpPSyH3VralExtOTQ6c6RNRZUlW22Q7TWVxCP1qsCCv94Y2AG1wbHeyrQY1cYjPQYcInbIDZpXyyE8r9U9+6ialGV9dv6BkWlMb2bifGs3JxvIaQl8OJPSlqIqA2Vkf3vkYl83+yPjcR3lUZnFgvqrYPFeM3iXrSCQ3uHjIfmRXC+jdI4exZsau3CfvccUYIZC5LjkvAsl9Wgsk0SWTIpDcp7VAPF3SKwLJfVoLxA9jXRZ+FwGVPm8AlmgCgt9lYIXPFcDy8sqj+E4sHmOU5mA9weNLemzDY7XU8uhGsxTPNS4KBhQ5tqGJOTYx103MsYn5uibm5SY+wjCycbV8oc+y+qQyS2RACoV/RBh+QFr8F+hVSVnBVKyICB5TTG0Df0HO6IXoSJmtGh0kssXqWeMT/k9bWB/tl1YhF5quSZhOjHYdP9WqNWSBkuMoz3p3+IdOm4BC7a6sSDiiqZzB5JWezK7IdWCMkcEphqx3DKsYKldInQQSypw8Xm3W4PheMkThyEPdZ2OEFIH5ucQ//v43wTod0Z2ldOwtUYxl2NpjOnyVCIiC7fss3iE/Rzh41G412nstWHGtxvP95j4fYxlGBrRv2TrcYC7+8be/ieuF7GiNJrF2s6wL85X3CK9A3nwslH4sl35UpR3TSG8dRaT9iINZwohmwjO6s87GU87HiOKeGg88uTRVOoRKrwklkNMxBKZnTEnFhn+gjJbbivIFocBNWf1LNHOgqMJwzcgICH0isiejDB0Q4GtBg/KIKhFJFjxxqm80Jw5nmdtUpqCV0eV3kwD4gM8J6gov/2NPpazLoINA1wvENA4x50ElbMxtd4DqHIxQhhqtxv5XLUfn0INPJ5fnjW++arUFRRyTzEYZQoRNswgU9f7q9Qs4MN0HcQhU6pttGPHplnF4wtki6qnQQuYUVJRyS7tL1Ats8fMcBQc6arTBXq64QAfeNNE6aUw7oHRCn+8YOPirIbCfO4ywMh1VPN3ac3NNBFnBWV+HgOlwJnLgMl+PJ+iSSDKV+fbR9T5xOqv/ajFPMOLNT2joXxX9IuNe1uiscKzFdy95gjYbhAE+g1Saq+lGnVVVC6uUVWyjqgRdQBugkaJyOxN2hrY8MOtgy/Zm/PNNrTY6a/xLbWSaN1xplYFHSBBbRx7lh0naofMk9U/WzA0dTfKNpBdNA52p+UAr8mOM3UU2/bJzVw6s/DwZIDgFNOrHgYlWp4DgIpmmrpT9IfYGmCEQA5bTGCpT3kBZrZbkcwFAT+bEhA9PT7P8D1C2xOOpfkhWBgZVZWWAL2xgIIsuPDVn0ylFdGCQFbRRzgOh3BHIk3KDjz8AdPJJITi3WrPobM/QloX0D0PNGV2a8w/bCjcF5txkKSKAut5VU5ewefdPyNDEAevTaDqTFt/eMAoxE6v04+aUAjof+G1dnJ6//vH4COPHj16xrQrEh6P2vkpocEEDkwjMssY+XmrAh+2DnOGMWS+I4sbYPZrh45jioWIptHdfdvbgv32DL5WjjnM5DYSN3ZwB1G44mEWzRItCMgyYooqbCtE9hagelkdJvRtxqlimhlPO+dXhqQr4IjshAa3roEZkiqPMMsd+5hTRqOJTDGgZeiDgHr1q0GTswrQ8/flC/X16dPr0HNaVeMvmlAafWFRGAgNa0TIKIL9xOEkInb3IWCuj+LtITt5o5rG9klGnrDv/fxM+rEgL4CgWQuxiDR+RMQQTaHRyJwMZ82Xxo85tAQhQSfS83WGK3qginhbkqg+1ua3LT5BlMahSwXO2AiVZ5koYuQFoq2VrF2y3cbbdpuZmq7u0NpMAscKKXlUJgavEUeXwmQmKprBYGJsPkrrXyovbRLSviUM3Q9q3CWffBtS+s1Use14oWxnMvi6QXcXqbYhBXxfFnsvR4Gwfwp4/xMBiKdJDPtJJnVcKijODE5obpzbjoaZrMT0Q19ObHBdZFmuDsIBM1l4VHGgUVe73+iMyGswMuWXdrUyXym4IdeWSx1/IVqkDGd4hDGcuVgjLbcVKCVzeSgkM0lWuEopZFqKIoARLiMh/SyJjXYPIL3Z/HbvcyDZXpnBZw+n+Msf7JJxvKjnfIhu3at/djPpLn8t+/kUL8VbcjSPrV5lsN/GkdoEntT8aUolRfjSkEp9cCekTsMlKb3YFrV30rVY++5ugfTjz3akk8Ok0rwVdY+Mxy+XyOICEZpKKYnPaWyvYL5eEl8VyOQeEsidnxrsw06TibuvK6UVFiuCnkqMp/rW+qsy5mTFj8ulSzarYbcnFyW5isSRu3G5SboDZHkYNSmZXhURuLGWOpDXlTOFD+lnkxrmdy5pZUX1vbfW9TdX3zeqVa8ma9lKa8KJLB1LGruwgLgWMIeZMKmX378xl2FhsFH9JnuXX+fc3BKz0dpNnfdEsUtVO8ZNuqvRh+9aMZZrNQe5t1kru7fZNVJ5KDvRRphKrKnc6qLOyF1x+a9fZfL7ZYbteYfH4LJq0GNP5jQOZS0bb3tDEmewaFjm8EU3Yr/YPRG/k4dVNZLb0upjxFU60UCf4PE7YZECauwYqWzvCrd5/2IlpjT/36vyhx/LoznHEhskS+GDlqFE8cxzAUlZjTSmSE6nx+cGLUTfGkQFpImIvBNI9VNNxHMcYsUE6gmnQw/TnXjgSo2jmU/JBncjHkgoYDEfAFfDj2fnPZ9kNDMI+fA5YwGgm0ENP9IOH9WkLOcMA6sww/WAcz0knpdMqU1ADp6rnS+g88ebw5JTC8LHHEVFHFKfkN00mR4LI1/ntYW3sD9kGUR32pD/yBk+EP0OPB7IGK1VU7EHnUcHjYZbiaJpXkmztvgi42l7oSwLJ5ki+4JWqLjMiP/aMEPI+jHnvwAwSpZ8SHg6o8RZf6YcVFJm52POPuuFKzz+W2q/ksE1atKI3H7y411cc4rY8jSMc4JSumSs6SfPkxncfkhtLpcbL7slRPCv5eDAXBpiebR22TZdu3Td5yIDzIeqhC50xvhpM1Xp3eHlJggp1ky6ZksIJ0qmFCCr4nQWUWSqu0FnAD+V6KlNsZTCwHkwBr6sAyGktED1pezRpBWN5Ip4KM+V9djsQLniVFETe8keX8Imzr+Uy9/jeLJpNTpMGXVosjRtC6rK5jvDMhCBGCnKnrj7l84eb14zAprgmxwgPj5tL8c/Z6lvNb7+VMxvd7an0HwSfjQsEvhRExg3RNwePJxoJg0b2ePJCdojhTmJalat/l5CrhxSgLTS4ZSWpIKYFGiHwnQX9WWbQOgyqQCZQn+hjz6CPDQAEek+ZiDkZ0ewT0cigNOVzbkv3FvF//zNdQRvaE8rOUUif89ABkfCz6e/C3VCeM1R5w5ybtXG62y093XjGkTgU5xdjdBkwFwD2arTqiO9EO2h8ZUz2Ps2CDAckhXeD/Zh84gWLrPJSJMKmwSJPFWNS93OTavg/GX3oLBih0ozu04zuGzNaVUvYwSNyW9jMTJSMyXxGk8lOYRX+YH+K61FUH4Y3Ar7bXTwbJCuml8pv0px7g8FK1XnmFE3ZEqbRg3T2a0h3wD3nqXo555dzfClvlkHfxGqFeMmForMipVanVsjSYvr5beVgzWjUOc5UqkNaz7fyrdZVuSZVfd76sKrfZq1++5wHAEZ8Zxu9AU6fC3PNP4YhrlNeEo8cZ/VIR+pV645u8aQ7Pt05+nxd73/TbNXFs71m60avwGeoTYUicHp93iocjShWl3HA67pkEHZW2MG3EjdjIT6jhbiSfvUKFdcLCX1ZX0gwS6TqFRz3WZHjwjR2qOJ1DX7WbpY4O/rN8xa9+TYr8y2WKa7aZ7RqnxmrVmJu4JoHKd2Cq9HPlvFznKIHlNlxOetwARASt04Izqu4/+C7Xc+vFp2Kq3lQHdNQvFJhXatErHc767S7a5bvmtW5QsN4hz4tUswb4JowcqAyRT3vm4lcxRid6T4ksXolQelh7YhWkbL04YXP8XTJApxvMOQSOLmsWSAlEx7R1HNYrUwBjkFduraeHTvBuOYp9uhlW0ATJg3JgxjR0No+FglLj5bN57ufTs5PDzF4RVIV3k/3cVS19urFdfe1fQKaqu/V9z8hXdFdy+uHtYrYskukK+hKD2yRrgxxgk/LBeLSRIdHgnumEbt0zIUj9hRvJ09eiKP23u6r/Rf6pIwH46TiYOxk14sefsXHi7xjakeMoiQl92lhY/A4XsMEh1BvpIkFh+sr4462xjoQ5BUmgxXIoVdYJ2dvznNDIDMFoNsYJ6XrLHJCaA3v5ajl5NBaqU7NWRp9+9o87xbixuVJdkonQ3GNcSC0hcYyiaIhIBUjfUvWJu2MmdlMLXnpLmCnkxrefS0tldgo+UUtrKNXOBhHyGWsc/r/1TH+/+cLa2mM9NdVN3+rU+3i6FX96LR+flq/Oq7/DOfivAz7dZGZKU+ZzgIRKW2AXxOz+tpgUZlbjKyhR/gbGmEjIJ5cVoBkOem3vOhS5gbnaTBO5DDSxz8dX/xSui5HQmfdl9YvCvts3zmQEQAIUieA4ADLF/LeawJSS+Qn9NdnbLoBJvzR0L9lygfhZo64oBcNHq/wBhleQM8MN5wsbUDSGwZjjzVxYkRJbaEfrWes3O0HHi3GfFbyLbM1BC6l5snrb81zWMA869qw+UATWK3J/nzw0crCqvUVXVSCg08LBmUmr2+QZPTUqVkrzBh6ZtHQYuECB8vsCnTTOE6MTPfuY9IT7XDJnmfGnFl565UkQro+G1oB+sqo8INBa8h/BHEk/Y1heGkwUL7GH9fmcN2g/EoXnOf8XUCM9hLcgGzU6oFMjjpDw5QnkcaxNlqiVWTij98ajF+e3mG/BzHAqLq0jHw9lMBVKZOAGskMoI+Mz2W/cTnZxqHPqRM+4yAOYI/K+ogrIKM3CSUbmwVdMelkfgpb+ZBlNnUd26IhIr/jgS4Nqh9H04kHpx1jdEujWoLYKaJdTlefG3Y9VAX+D32l3cgugHOW9UIGkb41gKFUxQsIOWao+QesJroLEbnVCvw0E68Xc/EDr11gWgwcVWdZ6pa8S281vmxWyJNEqcesWzLTTtEk1rJqtSyB1bMsNS1uNQc7f5W7YEoa/Mu+jZyTBt0iLVNY+1aBLaR0UTxA+0bCVtDea8iEpcy+aXTZrmblc02t3gzk3WSV7D+Je7ulLWA6z2SRS9a9yw75YdILUSkXVO4wste4ZyFGwWMvAEQwfs+PejOJ195/7LVQ9pMN2BQkyou/SQu2yXAwy4ArL1e7C4IpK/ox5wz6GudsJD4aZSiJrdP82IxD2Cgs+4vjd+fuxfn5ldgVFgqBu7H3YMFDLmlNPsONm8gLJM3NDd+BLAYcN+dmRTpHecPvJgOCSuJAKWN6yb3rzdLIri0Q9LJWF0N4H8SdNJ4FTunCDXY1m7pGWxWBgcYoFx0yojtVl/GFLUbBK+/El+0yUUtyUJWL7QlbUYJTtSEbgiDjURAG8V1nQagtEUhnobBbVu/NEg6NsmnNkGjCYhEKHLX3EqEKA2q2/eR4xScbA8U+dspXrUgwSH7SAA0bL1EYHvLMo0O7JaVDNJWE/gzk0BGmBoe/tbPDmvD66IN3dth4SYlugOzwWK6vo5l4W52lJZnmA4jokm7S1ZlXc3eo2XJMUb6XT6tgoUGItJDBJwKYDAE7OHDN0nT0YTC16a7q9N1u4eStHHK0kgPzi8OeTD6d5A46asjXnLIlOPfs0O0FoxHSvVehuDEgsc2uZaprZJU8bgQOhiq+D+/pqkNFDtlRqb3C+toPOWji4vCtsL8jZzNVpjtPg0Sp7sM/0CBnb8lYSylbbpwmWoht/OMiMOlEO612vy9iobhadNdG33vChjmZOYHVNlhVuAw0f1Btt1dNHELgKp0F/s44Ven02m7znJmsqYSRBEIQDQZlHGzbbHal26W1iHWAYdr0Cq9toNSMl1cX719fvb84BB4AjQ0ijComyZsj7aQC5Kyt72yQPl9NvrhaaTCmIYgTqF/PfyYBjLIDEJtR4SqXh2+PMzO9kgDwy597z78UD8NIPOAlpHTzI+MrxRf0xCBQIJejuBBOeqi6Vsos2zhtBL55AWU6xBQCCSrwZz2Qb9DZDMFPMNBJvzOTXnF+y5eUighEmNEsIW/CXiri2QgNsDsqIakM3MHenZ2fNYwxNcBRzBNdvBqmfKMTZi3oh9BNunC18VL8eHz8jvurxlrJxuRlQoJc4zmuzjSOAP9g0ot8SpdgTKtnZEoACX3E8/ZuQy45NW2FHHLA8DkVE0yv/ElTWtfONwVznEH0VlGKqXQBWaHrrKAumbDqI8AwFX5MfXsNHhh5a1c14FSYCN/RlsE6NdjI7A8YXplei/+q0uYzVZEvVD2V6/FSkzZL6li3QQsIFyvVbGTriUT28qLCLU06N+FFLwBC3auHNckbjkGpw1sd98Z7jAfEbcRTjhtye1J9wB3K7LNJPMDADgxk7YWutXSKqMuP9Aa+yig1mf8WuSB23MjEYSz3ml44NcNXCflEN47ugkmeO2eTQltrCd+FKkCiI5pBht49j3k2VDxQOsWgnR8prdPzTIx7o4BcwCrxVnzbUnP+ClN1Trg0MQ84cWscOPUvnQQzcBqaBobsqFVgjlI0VKAppxvQJ+tjkEMSc1ztPlHmpMB59lpN5T6RQWY4eTfhQruU/m2vmP6NZ4u73REWMVRyAiOOSfu0nW+maB/Of/2uozvmiF9zkr+wjv/t9en7o2OzAVOY2LNyl83kN2JzAnO7krBxqnYllZR2OrkNmqlsXi4UigeD5ZcNGh09yrxXedMpRTSm0Zr9jjYoE+eSiSOr52r0O8bKrGejR4XwC3zPTdwSKWvBU7Q0k1lbtJsFBi0WdzJ1vFmZGOXzuAnn0k5/Fkdf3UR12DyFFeeCt3uuzABTGe6df6Xddy9475drf+bPzThlSkjOjr2jkbwlXoUu22iXMyK1OZL7oBoQEDPfc9Sdi+FszFqZkEgR4OzXxetoAixvr7X3VaPdarS+dYAqOVpcan0QohH0dosxypzqeiOWTlP8fIEoyKV5RIBtPqE0WIU3SYNJIq8MJ2VY0PAD9GiFfQvqH4rLk7PvT4+LaZoTQCbp4yK6Orz8sTn2DZ/etrCtJBiBLCvOz4o1LUfuQSx/IuP3YEzu+R4nEggNM4DA+55YskQ+DftpIw4GYYKqap9FZBYD98Xp27cIUPlHo/g5hjHDZfMQhyl0E7rzXo0YgyyNGgwtRaPL/NMvO/sNpXBtKAMtBTlrr8w6S9NyW5NSMAC9DyaUEAjOvoMJy9bawks7chOmXt9oD7M7CqSHJwIb7gmbDSRDh8P/887OnNcBlcfy8mHrL8Rgy5BxM5lAB1lwUTNWoER9cdKsC+WHe9c2/KeieW5QKMAsD+K/CXpv1sNv+QYdnYMCoLHxq5RfYlE0ORyUApkLJdxR1Lujayau4llVdAZ7BOgAJMxrwfd2XeIhwAxM+jTRI7+BQBn20XrP/uB9Y1jEIo/8kmiG7kLjJAyBwLjHpSRVZU8GQbRDgyZ9zMwtwUiMtmnoVpVcNYRrh26nMrarcFfHdkNXivxCuqbL+4rhdRV1CnFYpaq57zkI66/B0/sNC3Kbb84zQRdpoGjtepdRRJYjpEQb52env8gdpLCtlDaUZska9IaZXQdzxah8LXK/YAaZ8ZW6WKhhq1UNW+3GWe4W4Zer0CRh2XJPcllAgOMuFBEssxwg+QYsHQiqeG+eJbMQO2l0wwmqA7bZxAoNyD0t289+tS5XbWi/brejFVrIb3CfZnMrNMFbP4m+PqrNtt726/LqzYdohoM5wWtSisC7QfoQBCA7zwHdMXQQY5UAKs8HEKM3F7jzwTGJ9EYcB9rQ3eZ90bBgFeD73vwff/+bLq5uhaT4llGE18vbo3AcpjJvyxdIuyeIvF5alDUJpfgiaL4JRyZYjaHTlH5FRxk9FXBe83exNgZoUNQScmK+B9I+eu40M4COylD4OUTtsRd+JgF7NrGNC0uAmDBbFsZlkMAM08ibL+z/5PiFdjayREo9DV9DgrnEgtTlB+XSHAS+S4ZwHOOBraUQ/GApt2S8IQUVcPAnmFC6KVLAZReS5O456Vhw3P16r+LLq6NXe1/rRcry0v8kMJ0vE/iVNQDP8sYTvp9lXTPy+t/4dw6sqVAQOurS3cDzDSsi7wYyto+DcOQGMevd+V1XVrAwrFcmvYik//4Elpgtv3fkX4kJqZkQIgWRDhQurbZLHT5g40SqLnTJxKY00hdqcJxkck+XakgAJZNCtSaQBmK3XLyZaUsrM8dLstJ34HwoWiq9/HZYFe/8qM5I/7E4VWzjFbgUdcZ1jgkutsmxmax/3ji4jnnzyaZOa31AZUwrugOx1f9ALgQrWx8oEMLDcjuS81xmjGWaw2A0TtHmyZxsFJ+bjbSL25UCqOBw9u2kCcNuFeKdvWsyaaN0RhiW5052Fw4qlIl5cXcg7sn2cFeHHyA5eyqrCx4v7sS/wLmJQG7Z2W4mIpa629Xd7eruXqzrcQYMs9Sq7U12vLuxi12+Y+JG8hx0G+o62/IJllDLnchfPaxlW9213hYz2XNNIZggyk4p6JUdWwODr++WFMGobRicHo2Lxn+7UfFddVtaeWDQZbw0Gv4Wo5EBdY0LlWVn/I0d8akj/od1JHBlqoFyPwKNfHaDmbkZbpgXDVkdZ2RHgo0d4dzogQ57sjhMPLG2JNm+a+hjy/3qA291de7nvquvkijek8PrckX3co24lLdbdrC/1fDIdjlztc49rUBIpDYOVZ+Gqv9hc54lfkUf/fIADfRYDLZjT3mAsg+DjcgPymxpsC1bKqbyLPcCc2AM93RXVuT+3NCzYY7xmlNEKTY2T/NQpeeiRlXdvY1jMyyPzbDtaM8kksrzOnU4gQyi3UnUGERQZTrDm2rgn7KliE5eQc5YaLYnZVPeV6M7FW9t7pLXlrntutJIqdR5H69DZIURtQkYlj2ES8YlKMUCcqVCBi8wxlhthRC3gaqEjTGvKs34dtnFV6YTnxQyho8fq7M2fIIYp+p81t+9nKxKBv3M0Xmes2DTzJ6uNIFwNEDWi8ri78/Z4JfRBs6BmjR5vax1dt6AglS3S7o/POdpW6L0INUgDFFIF7bkFY6+LCbvpsB2g91sLJnpWtLNWAJW2FTCNe0CrMD5feaNMHFIYuVE32srv6Jo9cGrfJmBJy/vMNSklQvjIOuvocdjVDMMDjTyph6xGNGkkyYRIZt6QTlTUED+Mr7JQcB7ePmXVjuYXCRL4KNT0divOMTGjLtJWJ+BekhyOgoy9rJR8M9AV8j+Rrudddl6FLeWM1G8iTR/RMjqu+SkLTmv8X7zwcDAGufbTMuzSjSJZiDqhXFJBQDvkSMDBufvr9zL96+OTqRgIWs0x3fwfzyr4cU9HdKaC3J4BsqgRy5uqxZ2ZZSN1in/lgDFOk0aBTcNHlMb3zR9wDmx9ZiFpKrr7GGOmr4H7zoYb5E5QXWsWdpvfGOVW3t3cf7T8dnh2evjZvqY5lsyrNsLOhD+Ovk+mAQxqcgWOE9Lvk61O2cneUkZzem8CVwi1ZlIfsXUH2VctO+jO+5ql4ZP5wQJVdrBV3UVm4wJ5fiG8irrlGk9qzBTcbY7LlnMYifNVQg/XWX8sngddZHjoTfKx7UwRtY98sZd3xO+f2AkIcV1P4l+9w7E8df7bXPi4uta1ghGtLM+fo0Cv1hAq+utXEJTt87xlL7fxGgUFxNgBYlt2vYcPGzHKPLYjro8EFlsfF3Ij3QDbCmL0Kxeq2zi8p7jMvR8FhU4qFflt8GYC7rX1uj/F+KQNIHq5IbKeSZm0zKV8+RoyHDHWybxWyH+z/8WhvoQvxCE20K1CvffUUAEfsBeqUDnS/H2VaGa7Tm5zVGH+UAtvdcsX6xIYAOF1HjVvNrNda2YzwfTIBQa7DkygzursdlMpVKoy2kHuFqMvK5ttGohZeXukMKL0/ujcMq3RuSAoa8wnuIfwgRQhDIyvvcGOomPUxwKOuabSHBUsPrOKoGak3OT6VvUYmb7KdbmDlCTulANA5wKA+Q72b3p6t57wMXP+qDelvvNWSSQQEI/QNOBvoSar62n+w4NWIyZzB5DRfhKxFo+WK1v8UC+fI5X2fdzyPDFHChewGIduQ8BZuEIfMCt1LOBo65tlxMT9eWMSxuIjDSjzT8f9g0DQHJKiZxKt26o2yw6txlXvRV2ZmB0Kq/jMIx92gBD6xUYny3Zt/MhreeY7i3evfZ0v4GxEk9lfMNH4pF2K9GovN1celxiCJzOh2r4BB1U3hHwAo2EFHnNV7ThpudUtJhzIqHbBCcgmAF/wr2yRpsxsgR1jqdipNUA0rjNrxy8ZiG7w0CEZD7rsf8j4K4zTNwb1zyMYI+346iLwqM09gFHzpISlHAuS6NYfgFMnVcB57kDyufx1u85212tNOp5yb7ox9C5XeDRs5Z/i13PL9lFjW3rNc7owMfQWqXrQ02ejGoXx6/P376FqaM0GyxEyyLLF3R1R37xmh4SCi/zHWJV6BweSZT4fyCePFnI38snTyzxVMBJV9jyBABLU/6SV6urgxIf43IuB0Yr+sidd0Oor65hPbEPgSopBlslgPTEbbY53sr4nRcqrEvKyERU789+Ojw9gSE7PsIwh3enxyg4v3t/1XSeyEZuShLq5fu3bw8vfmmO/bx0aoFUycIP7fvOSgkzZ9hbRTFoI6T5YONgpcpCvs45u9xUGA+/hw6Kn44vjk5eXx0IgienrqLwNPDuMD4ICjbbfZIOBL6DTWAM8oLt5FLAtnBy9TFenuNpkvfknThoJ7bzlluKA/LiARxDkqB5GA8ozPQdPsW2HyS9OKQ12amyoSov72kTL372ZGXbapDoD8jy5TZJR0VLkrMU3mh9k51GUGWwEg6ZxPAU12MkkjSKUYia4cthMJp2YNa6sxD2c2ncAsJDrzyFWjwgPe+0SR1E2Im0QK+yWdOphcYkadJvvEbcCIS1JOJWzoNKWcrrXJHwRr9amBKXlEeuS3PiujgJritvReLUrZfkG3H8GKY2T5Gz8/8A73PrHsfEAAA="
},
"02_features.py": {
"sha256": "bd15961e86068034ed0699aa26ad69948140b8c05fcbcc3ed5dd8da22f324c3b",
"n_bytes": 52514,
"gz_b64": "H4sIAAAAAAAC/9197XIbR5Lgfz5FHRQeddMgSMr2zg60dAQl0R7FSJSOpOz1wIhGE2iQPQC64e4GRUjLjX2IfZf7f4+yT3L5UZ/d1QA4Y+1d3MSuTHRlZVVlZWVlZmVldTqd97dxmYhn4r/+4z/Fy3xxnWaJmCZxtSqSUkzzQlS3iXjx6sWzP4oxFx9M4yq9WSWirFaTdW9v731SiLiqksWyEkEO9fPrMinukokYjZK7JKuidDIaiROBZRIw7CK65aoCqGmRLwD0fjQ6HI3W8O/eaFSliwTq5Nl8LQLswrLI79IJ4SwJMKZ/Jyn8EnGRiFkCzcel2I8nd2mZF+v9vXGRl+XB+DYZz8quyKAvMJxiVUKjYX9v70Ask3gmymWSTLry7/Qmgzbi8TiZJwWMM8+6Avsi4uv8LoHGqz8dRdMiHmPnpkSc5TxeJ8XTck+I66QEEnxMq9s0o7Jxnk2BVgXgvAY6z5G6c+jHHMYfZxORTIHEFYBhvWVc3QoYUBVn4yR8Dv0rk7KELkAX52lVpdlNVzDyA1WSF5OkoILxarGaQ4/vkFYpTNw8jyciUPjE19C/cb7KKuz3Ir5PF/Fctl8+N1P2tUh5WqgDBQ6Ixk+TREOa5+OZCIBEiAyaJi6RldT0llR7DsM60JyRrRbX1NeDm3gZcg2iMPEe9Ranuyum8/jmBqcky2FWCkC/wBEgRjn6gGgIc5aXKWIIBQ5yEhcwWkap5qaAYRZrsUiqIh0TyZHt4gxgSkQ4Gk3Toqwi2UvkJYBJqxLg5jhdcZFWa2pXEvSn1z+EwPSvEuQVkZYC6YHzuxaTZJzirAAT7r86/qeDV8f/vL8vgtEoy6ukPNTFvQWsh/C5QIZc71W3QOXbfD45RAaAf4p4gb3E9ccrg7mot44Xc+hgAGRZxDcwHKZoGfb0yp3EVbyXljTIeVIlNJopDgcY8uL0bVeUuVneuGzMKoRZLBLgCYB88e7qz2KRTxLgDCDjHv7Ja3F8G2c3UFUtySQj5prH18m8J35oxXz65o2erXIvgDWJPIC9Q0yvXrAwEeUK+LBCqiYL4HioXCZIkCqBtpU0IpF18I2Arkzy6RQm42KV9YG9hViuq1tcL8X48OhZpAbaW67FwQENooyRMI+BhV/Q8DjZ23u3qmBAJU5ozn8eqlogiULugRSSFr64+G0FY9L/WzrisrnuujVi+bBGLHst3CgT8NvBMl+SHMChMTWDGsFCP0qWYxZO6OgBfwRMC1xHVFFWiCZpfJPBCk/HZW9c3pkBwooAqTBL1od38Rz2CQuQEOi1+om6GRHHOzgCkLqLJIZ/y0lIBKuv+K5c0jyWu3QaFckSRFmtJ/p/sGpRJnwm0RjhSo7WIGGcxf9AuN5fvPvp7Pz0/OVZr7qvuuLyw9u3pxe/wJrd63Q6e3u0JqNouiIiRDhtKMLjDBZ5zDO2p74VNzC2MlG/y3XJ1XGhz9NrVfc9/OSCar1EiSq/n2ZrjQvWOnAm7G/ZUn1aIhlL/Lac7O0B8h7i7aUZsFQVHAHxqiJA3AF0N51DZ8MeTHQ+v0uCEOcZtuYw5IaBGxbAMRJzIMQTkL6/xX1x9u3RM6JLvFwm2SQqVhmQ76YreSjLknElf5R3EXaAf90kFcLdJAX/JpKzIOMPuN9GMPHR4lp++E3+V4sV/i27zD9K2Ksjkpu4EUhUuEFNYEnENxKqyiPJxt29cG/v5bu3L16fn0VXF6cv//L6/Mfo5bs3l6CPDAi4MyniaRWtQdR3uqKjNBb8O5vO5V/wDfaBCOYnwZ/YYqerqqfzuS7hX1m8oF+Ss/DPe/xnjf+UVELAacn/gZb3hrqfF2eXH95cXW7spumaWshqadjfbpP05rayv3yUXxgnSoSoTD9xZ4tFNE+ym+oWf30E8kYl8BiNN8lgvUSkh+Bv1BpoLDATsK7juUJ4XeA8/w2YlarB1pbgrBN+2OSKKipvV1U1pw/XSTa+xYVbqurZTRnh2kgAaVouonKcFwSKBbgjJzcw0CKfrMYkOpzyaZrFc/UJyLn3RLDIxr18tZDKEmkbbz9cXsEmWo1vrR3lGaqCY5Hcx+MK1U7cnaw9jmQF4FTb0MvLn3h7FEpkxnPYXcdxUaRyhxyNTH3YubkboD3AxH54e37ZF3PQzgbVCraZASxWWrHDoZ7yoGOq4yB/Or14+efTi07YlcWGCV6fX539eGYVudzSKPYxTQN9mvE+E5k9BeFevHv35uz03G7LXgENNO6SaBRba6TRTVkWrTLmu0Zle7U2+19GclP1Ipc6dFtt1TbIFWBbXvGic/X67dnl1enb901AFI8bwJJ5vCxRTslmaen/8Obd6ZU1oWhbgObn7W8WgeLsLwJ9Hgu9OEnQkpEDS7gVgEweBHjmgUCjx1eTzYcIzReo6gFAg2hz2/Z+vLlc2ire8VtgSqPaik8DbkQM9kf0aTPRPrVQ7NMWivnKHYXE26OCSOqpygUgsDdzPZdFZb4qWqQK2l/ETbD+LeaBDWGSf/TibNRgwmyrwXhBl56nSdEONgH5CHsMdHoTXQgUcEX5NCJB3w41B8Uswf5u6Fo1iW6KfLWMqjyPSrCV534BJZ0NkfJPgFoM2vlq4ZmgJixTd3fQ2AGFHe79m9Nfzi6i/8/2k0xNdAnUr+J5izTUQGolb4PbtCwyJZfbZK/Z5DZDZK3yG4X0VmFMA5YiCVUue2xeWaG3E9ymWraeeWyECm5TDTBgpbdnVxevX5K+6dkyPHuE3hRqu8Bw76+SIQHXZ9bragj7QgtWq9xC3xc1Actg3GRfKPHKX2vt90Vdyu497KEyePD7/U88AXzvwQAUt8kchlKKgE2n+HqeoCuoAmFckqqJTqx0ya4/mKMqCX/3rkySqfhEei/LrIDs7hJsZFhxWV+kYOexdwIs2L8eEKQIJpN8enIUSp+aiAXV7YmLBCzbDEb0qSvyGfq6sOZolM9AhU1L8QOouQlrx6PRJ/SYzecH5/E5YLpN2O1JmBAWhKcg4cmMOBpl4l+4V6NRCPq4mIBCn6GnFTCWE3FyIo6IanV/HuBaZTBOoOIEuiTHwsY/sFm27MUlKN7xWo99gkv1BAqmsNtX//QtewoWcTlj+LQEZGmVBHdcksFnIFSAED0Q4EHI39OpwE5TEXZcUpItU6QUIpuu5vPgrlfexsukix8ydF40utBl4nFPVtAgFQR3A2x02EOXh2oWiVErhj1JTZrpWl45g0F/CZBVkrJ39Dt19hPTbMeaXIV7DRXVAMQBDDoUh9C7PatHwGZXxSqBJYp8vEhBnIHlqaSV9HFqR1kXfdeqlFlbHHzPGx/8GGo+P5UuNokA3Zqfj3s9q/YD8GpciY8JrIbzd1fGBx6g45p8qWFPcdltfJegPEM+iEMyAmPEqSo92EMaxAagQIdpcNyVzKVbD8XX4jjEOYxpFgEU2xhKOkySKhkboY0GUKA2sr5YTnqvoHs/oLHAyxy1P7RTrE3IUMcG1wQ6WyzTAm33+Vq2xjarcoUL7bAX2Dq54UYBqwgw6Xo7DEfims8EXh3/SYmLU/kN1m0s9vfRlTgyXRsJ2d39fcYdjOyBnjw1oE9HSKOpACAw0gn3/v73J+IZnZSkGfTaRlyK0/NXAspHoxaq6IObBdFbCjcADvTEwNI5Ho3295+LHOCKjynIu7TyjAWHr0ZPxyFJsb8vRZ1vQAgPA+qJS6DqPHFJQmjYdY51DtLsLi7SOKsYIclUnAzYXZx60FtPGXWNPeyIkb2jeEBTyhE7XRppKa94jGU9MoReFrLogFyxggxV2jxkzSZnqL8z4lJ9xGK2GDwQLPKPyFoSiZe/Rnz+wu6TckNzNs1BPtnTXvvJ6hILmugYx29vKdVtoXYDPxOxiIOul1LnnqTjilXu02zNKveQveUgB3BQLCZBqN4US1zris49mrvrdTCwtG/LoTLEI5uiOkERaW091KNS4NQvNQrb0xIOtGMFNhbg7lDXBUKozY7R9NADaJfT52MJo4BAh4QOBSRTcIWE9i6J/6OVfOK4QFG8BdQeLVnkRo0dvgCdQ5GgOmEP2WxZQN8e+52Dz3u2O1/RikQcklequrrcQtcnPzhT3wVyvEt96n8NwlXq+0S6TSAuW1EFHq2p9BDaO4UtmQMcb1ex+clgrz7abt3ZXHePNWyQTZ0bhnKrUXaRBQvSZtnYZcx+AgTV28gV7hm2PKKdL6ufCIvRYtRXx7JapIBY4rWNp4x0TICmOWy+WhChvoiHnlPYp67j8azP+qQL8z3CPBfkABYGUiGl9fG0dLppdvbrZA4iiFbSAP/fWjc2y0sVMBwq1WueZAHVbWqEyHBU1AOZPqC/bLTp5B4UgSB0jwuGzBgcaLC1O9/7ukN1/d2hIuoO/VXvDgoIb3csFNQbG6KXIr4jzUjypMZRWspWrWU8vekLV24Sc9En9mHAENmHwf9qnnsbL/nwvH3H4E25yuks39Vk9F6nz2hVh2lnGwGfjkaJ0o6Q+WBw8WoOVC9WGW+KoOpIpQmsk+cSn8VeI1BzR/ZWTOtinIAGMU3vgfNJWOJS4XFIDYrNxkN51JWhL+UurdagMeAy0w2SKjL/GK9L7BFso4C4zO2DVjGGJcibvl4G+1w/mezbKgsGf+DJPo6KJ5u0BIxuWCQZqMEJyl/cqwkET8bdzZLVgRNiEZjTQUeRtTPUXMaUVTwF/QD4TRoua8ut2NSOjLuzLfGGcjdCXNECmAQU9oB2iJBPQ6sw5O1gFe7Vj4hxMjDapytWuEF/Spcw75XeloEH6ae7N9MnRwwPwwezJJEy/wO2RM1NHbM4gcYJzA4DQYP1gxKrIej9tLPKZln+MRN+ru2Lz/Tfh46zHysqgDSUahuMcobNqVLHcFEf5YKegbm9iJGfIqinpioAEt13xbor+OSeV7Feu3qVvgdFWUlog4mDWu4P14cUV2QiIPjkg5hRWRE/8KfFCqT9dcLndxxqg3WNHrm0WuK4AOAg5NYyzjB8hxfUDZACnbtlVweeFOxRgLV2Q8xOLgSK8IAv0QSU1a6Qf4OWB7/kSscvd6YwNn+yGYDxPWJVyuAXbKn60xGQqyztkWGPgHqxbp4tDPi6z74UkCC4uaFnXa6WaTyu8mI/uV/SSqYu8nL8EXDBvJbpJOFDySSeHZDLTaOXXaES8qwJ9tCThSCjrpA0cllPK3Fx+jPobxQ4l6J1s8znMcao9MTZm8uzn/98dnHWpVEIOtzF+LrbOJNE4hqRPpuSgXocVQUSx8YI0w6yGnunhgYbYDp5TsgJH9l0LSjRenewoQsJkKWFTdrkfjxfTWTAIQsiGRx3SHQ6ZJogb5QJiGHYIJzTgoaV4DqeqjZPyL0Ld98Gt3bh1m1wqLxXpLOr+AjtaJUaqTxElNqxpfaqM8S+ct6YEnOE6Cls+G/9EI4HtwHS8NM2IDafPPXF0SZY58zJA2tOfdxCXuncHab8YOhzh9VqAOs8ts7dYyvEj60gpY+/2nWeKzvpwXZtPmsojMBSLKZgMWo3JAv7QceSPVGptnScCLaascp+rUpDgKlqZj03mqkvdbJXaNHKdTdJp1PY0PVC5q+362UOPC+L78OuBl1L5YDsDfiYFAX544NJikdsJx3QeDiaBD0v83Siv1gatXQ2g3QuoCIK56Mut34IHVM+UanJW07TSbWb1xSmZMJrmyxm0/AkyWDbPAFdtBoc94fia4Ab9A+Oh+hQfdY70oD/0PCo24TfHiY1TSMN7qjwQNzptqnUDFzvtCdC713AGEpSDZQEGkrvAvyw/QgEYs25ZgugCtlMsoXpnd+Lny5LiZgWwA3W0QOZ3nUF/N8B1sumYUje1+ldL87WgfRDHBwz+riGPlbo4w3oY1AsGujjGnoT/AYLFnZ/2GrIcY0adFJJN40ty1pAADcOFrjEcvJrvL14Mgmg3IKNa7B2GxI8NlqxrHIgjmvV2qoiqDWHm0W5mn01Atp0UT4hPwzg6zDchMsR9QqX7iFZwhYEGsvcAs6EbmU2DI027MATJslofFYH/bhepXNWGqsi/luCUmzdt7UOwrHkE0BSiXDGiTMIhZ53aAR6rGdK/JvTOI+6km5F4n3yJvIqqNDaZpO/uG+Bubdg1i0wawMjtwwJiNQhOPQ3DnH4y3k8TjA8HZQ7JNcyhzEDPTEMXSpXofZ0zsyJhxRjlnCZzKzVjOS3mQ0Kvz+x9gOX4YpK+QEVHWZ4fAJc8hwI4Zbd22Vrt2ytyxz0kgYKls69XAjDh0aNQDoda6hkDqNgvnL7Dp+JLHLu+w3D84loaNh1RV4oUTBLkiVbLvFHRf4Gwi9MLQ/FyDvtACHzN4e6kGu1gOqTAOb8EPUFlpSoOOBCJUF53KgKEAvxL0DxJtb/pjHvNO72sas18jfnVHAR+iEJuojHQLC/AZUWrUCegQ/xbJEq78O6CturNuliV61TCyS8BRduwNukqQfvuoF3vQXvTsT/f4UZmkzw3yvEuDdyO4li17YsWo3V4r4B22qwFusGbKvRalknbhX50TFSUMmUXXe0fRyIbLpN48f+W0o/dvH30PsLR/EvlOZfSNW/qOv+ha38F4/Q/gu/+l+Amo3af6HV/+JL6f9FwwCgxln/L5QBUBgLAItrBsAT5WwRq0zNutytpAPQ8rgcKjfMXpPSFtsAte9cItMwSCk2vPUHR20v7gxhY6KrGWuO6gih6I3z5do6IM1noB5BoYl/kV+ZLvKPP3BlIoOZp9jufT6DTsc12kzvntWMC9VLUjJqrhZjAUENNDeA8IBhqIyXZ7Z5we1IC6bRSlxvxXLXNJuJB4BCNRO3NfPEXGHFswQ+StAzTbwIawW2bTXHpMejgVeKYIn3Q43Wgxc5tTXvcwk0cZhTKnRM1OIQ7SGhiUmLdeAyCy1l2T3AGDp2h/QMISIAq31H/w+XwF9u2R0X3LlfY/4au1+Vx2bo8vFe3RvzBeIX8SyO7mXhwVjwajWevXohctRE3/OFlueok83Tcapu0JRfJnaR/ZXyHmCw8VxQf+J1PAWq1U6JbMpZTlHXa9WXnDFturMsj1rTbWXV8/m0rLoN95VV1ePasmp6uNyq610Dfk8tH35Ih61G4AcYascgn/zQDQl2I8MODeL9is7NahOBgmBC27g5m/0hwVtVHJiuznQCfdzztUgnZWif5ATqCg9ftDfn8uVveKo4VT/xf5dnb85eXglTw5z9quj0rfFH3boqlyzz8W0EjEfNw352fHR01DsSp5cCOEIddZXk2wHq6+o/XLx7Kz5fPegPdBwirJuC0P2nTIinGujdxSugy4tfrEFAK865gmReIHsvuU/GKwzr/G0e9qZIWboHGqgTd3nHWh+immO2gEnf98+W5+iOT9zdI3r7l5nhN3m+ZDlRDw0rn6vOB06wWFdtCfwzlHyhpzrBPZ+/DcyVLl7KMMVWoTqM5X1ibZVYFxRkKUyBU27Obbkc2KJRzoEN7IOrKqtYh0vw0U9pFZHc4JMe+/O9/Li2P647UrFdZelvXRnG1VW7J23YWLJKYFl0JTEjsP2Te4oA05+4wokxOZ6YqhKm5BuLB/l4vCqKBGMEAFE6xkv9QPC0pMAyVMxyjpBR1/YzTO+Rr5gp+c4ka+zFDdaQ8WD6pP03KMX/DAh0uOeEp/EfdpEeKv9hF1GcgaoDooIh9tRsRLvE2+EemtxJKIvjXYj0fhsEMNo2kEm1FSLdBrG1p/E2gHHeBqG9cjfG44ATVffLxU4w4OBmCDbntfyGU4JfNDAqzW1hAeUg7l+DbLlXf6zVHyxtDBY1my3RfVoEyGAN5K6bYT18rx4DOIiHzTBALRQYbLL2QnmCBVvhSERIOPjbA6dkBTcJvzww1iHtbGB+DRtg8rx2Zk5OaiC2ijFzDk9qgI1D3FnT2vBVcU51Zx7boVapcc47a+rmtSrbDn5nWw8UtiGsnQ7Ptp4q+BDaR8izpj/YV8W949h3Typ0xK1lpnkmpel9wsMMDyAY6a7F1KbsOQjDRnAq9QlF8cwYQSbgQB0dl67fgCWvWtEwpCKB5lGsWCsYatXcViyOrWoxS6rMvsMDVPvmmacirPpmg7Y48Lc30Z44a4AesLQORkZfE/CuBnfnA4prQLEPaJzXoLR9GOp9kMKcTtzQYSVQtbtD7oN7FvJ6nXahC7QENQCv3GboQWJcdQlGE4eaib9Cet8qrz3QvhDuSsWi1IEnVQM2tWJKGuBpHfzORLHUgRvDjK14ljpwXAd24i/q0OO8udJcL7FnknQ0uJPqwNDeSQUykTfDFTm6aqhdPY6u6aQb4avMI+zEF3E3XGJ+lr7KaVaCoaWSl8GfVsqyQ846dmhlCPsyXgcOctX3fjESsR6jTBupNJ7VTarLBPN7aGvIZJFANf/pZ56Nh/7n2cPTLqhLYOKSYkXWbZdTuWEwLYdGYMhePOYoX20OwW7BbMBNBaz/Q/d69JeSi538Gs+hO+b4le6ZyAsmCI7qesQXIp37JZ7UFsPQXCPRgL6bJ9WRDE73YtFgHAJSHbEbPuyByOWrzWUC62JSBq5KiSfyiiRhb7xaUEwIHnMY/+uqovh1vPFCdODz7M5ndQUEaI5/3ocPHaLGPZJh5jiEAIe69EDMF2H+G+S6QF5gn+wape69UvezTE/HWbO6itdDw+lft+Tp+9qk2pMRoufu7Tu6XCcotRvlshk5F8woZr1+CaN+6U45Hokb+bZZlx26dqB6T7wjk2+WoHUdjEYUEyzvmtNMw/4844BXfaVLCjaOdmitAvg+tlEIIdT1EKz+tMRrjbwclJfBrCMRVJhjB9YWEQf6WrsUrFYlrzHlCG4ElDtgSiAqViAblf+0Twh0d0zxwE41M6Srtw4m/1rhNV7l0QR2CPwUbIYPW3DqvDM7YTTQboSLSc+oSMo817ecaDJJIM8D3e6QDGT4x8S6mKaJHQBhxzlV2jMHl/U+ciA9DkS6u5CkTni8p7JzM8VU7TZA2AdkgbiEUKk1rTsQAR1ZefiwDM14odjmGBRV5tLyNuk7rHEYHjM9VoCrugM7sxGS3LfTdd0lUuetGoYGVkUxdcsLdU8QtzCxKV8j1glKpzIQW9/TUdwQwhJvOBAJLRSqvqII8pJ06DuR916mNNwnNzQO73HH0yNppe/8L9C3TpkC/l01qAPpZBkfiJl9EVrAWUNXTIB3R5xeq0pGWxt2RRuE6bBlurQB1yZqYNGud5NUwQyAzvMskbFuFImEXR22ibqBnwl6WV5lcRAOW0RhsEEWhtZac5aad5+QWx1lfWxsE3q1Ua1yZ/bwcoabAKzBJ6T4bFFzJHa6P4fJFZOK9XLFSdxNKQLrI6UZCxikNeilzuJtIsEdjD95mcVZO7e3hXBD0tbIpGB9TcU/ZDN594jH1zOk2WUKwlaS+fYWL2EH1qE8d4YWhLkPBqIodAMKrGtgpXUNzLTWXLzuKul6dz1GPnRYn/3qMveK3nIwP7FXP9R8/0T8hRUz78DNKZcwq7Crz5p4VVGWQ5msoU3fsblMk2bLxDVmwU2A6WbjM6c8oZIoNnmcPGhKETmmTBDvXlyeXfx09kpf4lIJuAVeK0pRjbbvF0ustA/Fla3MgPbYXCZSswQNf2Fu6YfP3Y0OU6soARjJU5S6JNogD317V+u69iYUra1rfThlSx2of89bt+zihlQDrDU5+2PvNgGz6DhkW6u26tw0dbjaXIWXKKB7gQlDGxVkJHwynTqKdN1RDBvzfA4bz1HvSOooKzy1AzA9nLalaA9GWZS1cbjpB7FbiP4A0bdDqnyBQ3vBaOPZWRi1LkgxqVmHB28Mfa0bb2CeLo3d1m+RNBjVhU4C3R/6uWG4dppEqeFFfy9RgWCyuh4ZNL9VKbCus+/QT5vqRiTJtv6RbkN1V/bIRJ1aKtfuqdIFVUftjzBocQdzEBHyyvUYCoRk6OWjFiXESuthrZ9GllG/0aVatDlJF24xRIFwLUoibLKBGWXoc/uYVuzrf/XbFurZgcB9d+C58L87EPrMTpkpU45fRazVy52EmU1JNuiwo4i9rUN1Ek5HFHjRRMGFxgsXwWKckPPpEdKBT+B7KVC7tM2KcnVdZxZAPnTmUe5JAErl8F/fpuPsML55te5jgH2hE9DLW0cbsR+DiHfoBLNOlpBlwSyL5A69D4APDRL3O7RW/y5JifmB2J3JJ8ZAoQL/CGphq4yBLxd8rM1aWL+QofuSltRo33PBQVI8rgaAsMcxRC08tXPFBrMdI89LpjigTbJPGQOLZJKM4XNe7HCrYntf2f3FlGlZ0YokYX3BNk0FGgZgRQtYz96/iCX+f86vxpAL2mGj8B+kFEUo4FcXkcVVzuiMm6sJzqwC/7aqv5MiX1pHLx5m0kFaXMNNpUOc+pjEJ92mRqhSMfQ3JERp90S/2fT0ioyj2friivREX6mMjhQnqnGF7HtNS84MtzkRS8xrbXyblwmvmcRKP4cIMKOK8jYv8LK9fP3Em1RmFDb93fKCnT/R3CZ/91tOOagyBaqMcFbGQJ0a7kH8+qu2LmReMvnUDxHjOsGHW9hIsFKjGXF5coyJipweSqe3ZWDgwQnaKrIE0yXYVo3kv6UIWrIlyVG2JUqqu8qruPKnbyF7WSeMBjA0GBbJJI2zjhQLMokJ4zjRpZivhELBrconXIZskOLDKuhZxWSLN3klPiPAQ9jZOa3aE3Ezz6/jOdvBKqyQWyhNtqgQ9AV8BApzlEo7kecXH4ygLGiNPGy1JGoNe4TbkHoXoak2oOH0LdvQNJLEyeSJ/3CaOJmSTS0qywGC5+0mMVpYyzdjx3kwDtf/7u5BMnoQJifAzbQrbkK1e1PoGY3GS2C718MNV6caLRDKkC6ofrzNMecuUYzfGHPTPdUVCnk+yijdZmSSz/a0W1qtcaot7nXu2PsAcdT2TGyYxElbclOs0wU0nruAzLTRgvYn6thmVqopOnZkkUHVcucQUXbxQSRz88JUwSTQ/Ivmr/OIW487MZEz3dABXqHEsa2Jx5reu9065LRRURsmR+BGnDaJ5HW8m7BO6BvnnoxZWyyympRrj4T0BefQAvBDbcmK2B4WKSlsIqh1arcWDI18731mzsZ3M1dtXbFiL62Jlod0u1SWAZktaQzxim1LdTdOcxtYe5pIc+nDqKx9cdwC1AiPJJ5qQeg81tBHxmubehNPqu4YNpL9bE7yszlWVKNoTfRj0rNvxFx7EmQL9PaMQI/KBLQ5pFMjqwdtbsTQfJ7CwuQ8SLEFke/NifYq7S9PbCFp6zMUf1+9eGu9pr1JFtF5fGU5tLW9pstaVk/Dv2UxoLR4G18cY9JL0i1ZWcMvGhHn5rr/MsFu5tG9xBfo1hbmtPutII59EoHVUNR4pvMTPz5Qtlwf6u2UU7LRgrLJKT+/VMUatRb0+ohaYZmqo14IbWnLsoYjziyoKn7i002kWaAevDAlA4kXTRb11gUqq7VHYWvvky4wbUzFNnj0qRYL4p67D9plhnFichh4tGueatkLmEX5GIjRnj6NycNNQ2MwxwU5wHKPU4wMm/kd80Co/aI4HG3QSCbxqA62ddDqIJUXwRWVpINUvQ9ph/k4VehxCxxR7d0M9WqGC+1gVmP9VPcqYhr/fNZUOZ3am+fNzXvC+iRfUy8HTror+BDWXFtmstuVyY5k4r6kT1MSepaXzB26Cyz7l7HGnQ/eUhfhTw9AJpUxHibfVOr6RhFn5lIrw8rXM+i+uqlta+K+AUzqaOxXNh6Hqy5h+sxJHkiGyGdysPmsNkZ3N6JoLWdLsqb6i+5MP73+4cvsRnfpNOCnoB6xFzUdnZxxsCW2Ex+FpbWmH1jW+YdJ2svWB4PmI3S10+oh+YMx+sl5KQaPf9S5zy4iVqZb/IbevWmcPfcyefETU40/kzC1jrgwjbSNDo8Matc7qqRA1bDCOwHp1NaRkvsxqLulRSdWYDMkDwLWb080CYu51gvrcgOnvUVRjyZhVpLncCuZxdeipdeApCgC9/Fep3bYqY+rdrll6xh3GpdkIM5gUsVViemw52WP/u5JgwGPlKcgBenGLT+oyy95gA6BJfzqMScNIHT/yhsnnzegpjaeBSjtwdAvKVTJwy12AICHUUzx0LgY065AeuK0JFBKzy4Fj5igWh5M+TpR29CCf4X93vKEORuTZkf4j565uxZmxMV2J76n5yL8ZsFuE/mgDsuLws4Ngr/HeTINmmT20TYcAOMcD826bwzqMdyKoJ4x/UMM+3/FmPm5SPFU/svsF+OYXrqbJ6Bsy2O5lscO3fcplvIauqXk8ikj5gagA3rGZfu7ZVGgH/AzL+/16wpXpSd/+vTq4pfo5enlVdD5jG08dDCDxGfA9hDiX+rr000pwmood0cn57rT/TUD5un9LU+zYMnX9OXJ5EeYn0Q9i805PSbTxta7kbZderi8T++VdzmCMbpem8Qg5iSdHiLnF8Z7i9kkLQL+UcosBsk93lDLZ1YCg7s0wYOvDj1IH+GvjnpjvFckNwCfFAF+xn4PBmOazHFXRNY8DuWq5NQhwbTz8t37X0TAaUOQMp99nBQ+QBFn80D8+Etn6fisRglUdxy3087VO/H0Mw704akIfnh38fb0Srw/vfifH86uuuLlu7fvL84uL1+/Oxd/vbx6FXZCPRo7qYf+uMqcQeppm8fXyXyeTKJxeRc0Jwx9MTQB9tzUJoOvk02mjgW5qqyH4t13QfFH+PdOIyKGzQd7S4/By7g0mVLzSwietzEm15RPLH4R8VOssgA3ekloesijT+lqidKwPTGhQQXFGwDmpXtJ7Hl+c0NJNG6SKuIfQeh7zj7Qngss0OGcOCEYsKAnCWG69BSHJnoEs4MxEHxazgCdnJ4/x+ufDe0Yq5Sra3x6XmkIuO5wztCSlggPjVmun52QIqSzZ40NEx7nQeekI/bFH5+FzZIXr148+yO/sw7KrcIl/o0GcfIV/mWGB787PDybGf0tKc1a975HTFkG/PYZmuQ8XyY9sI1nfJuMZ5zS9jat+uKrkpm6FMEK8zUf8KstVa4yAOPWbdpCGWxltSC5HQG74CzLOTh6pknXN63p8R31YP2RIxUWdLS4pmd4Ot22Y6vPnXKWQjO40/Ma7BRJXJJF3TpVckjPhTWkjnXVX24eR3tqz/ytwUlq0kNldyXxpDSuNo4UKDAwj051Pz+E/GmyGs8m15GsACXHoZZ5FHCTZSiJZfmJ/G9oea/w/lI9P5hcPOotYdt5F68meHcMvYS1l4Ylm+MhxUb3IgJcA5+iOHH9i1gic7K22JwKQl+u48etUjxoLNGm9jFFnoEAUBxh37TN3e0aZCpxncShZruC9mYgPPCVGJq7w2aZXrRexoLN6sXr87Po6uL05V9en/+I3sxLKee6cr2Ej+0WDG01B9nj6ZUs2q1TsIt+eHN12d6nKxSNvwVbh87QF37oepfceFTltkIHLRip6iQEk/Crw59DEJkH+fSA73XQMzNW2u+8lJdhAlsBmHZkQjMVORVna3agBXVnaCgVlIsH8ePFuw/vUTc57jTOqDlH2AQvXJv7NupYGMOI2xtgo5Hc2Sq2Rw2z0e1NKdkwtHqSlqSkyhMperM14E981sRfYvMlbqRVa6ZTO1HJ1GwSdDqhf9D6xFyOZoK+VlRoVUBr65B+fn31Z3GDXQtq8+NLJ8dZqoJXry+vXp9blAjFzMfWjxte91n3m7BO6w29YfQ3EvkMbGaLQvF8rqY2z/PHECPemRgYks559Kqj32n4NelwvWtfNtxwg9VpXnt99xOo+gFo7levr1Bdh4abaI1FUB2FINBaxxY3pktzhgcrVbmuZRBcxuL1Jb1pdP7hzRt60FaH94Hwir0zWia8nTqz2FicZnqAiBjJgstQf0iydpr9/svSkbDW+2Baau6wd5pqze1T5p488eSydBKR6awnHAC8OZ+iN5UZgvpCX0xcoLk5SFck5H7gQdKSJ0CXm2tz8q4FKjp2gHsDk5MdYBOeJPPjaY0z2IRQ1aKEpQy9E24ZirAbagW8O+b4MZhjL+aWmA/3xql828hcNp3wi4Yxnill1l4UqvumuPur+6YNpuma5t37evXZ7gytQ0G3y04kyrbeUi/NJmF6+dguWt15IjD0hxN/oxJGwYTiXihdW74rjYmfdWAAvX1BNTQaQqLn0BcXqzV8vEVcT5HGrwj1ioTiw+rRSI35tlpaJAXIIKzQpcfxojw78bbaFUV6c1s5aTtv848nHaxlNfGEHgXEezsLTJJNIRGNZOkkBO8wG7x8PQE7YCeMesaJo2X31CS46aKanew2ArGGww07phmvB60e8AYA3/j59E4bVfs0mkGjXxr+lmKZGcikzf6D/HIn51fmKcDc3k6Bfm/6Dy3tqJqmfzw7AEyn99C8dT3P7GMDTrs15GesNGdxMNx2hgI4w0829XZjIr3quD1HvMlPjSu0Vj0a2b/rGm000cF9nhgPXTn13aNzIZxwRoqN6ThqgEn5BbaVSldyqLInHeJOfkh3IR6jJMhWsXJTS3iCF0/68jFaCs9RmYvdKHLOR+G54sKR575HVzc+h0wuuQ3bNqfAaYhpTyS/edq2LYjfft12dwHOEbgGy9C+HrjiK++++040sq7uaTOFCV955ux4wcAkLGCs0BF+l8JhfeuqpCf1DwVe1N097v1jn3Z2YlxI3l625QiTc+ckDXJj+NqvC+/Ar1ZQTZNf8R6kTE2z7bqxTlCADDKJjDvNjgVsXENG5MMad9JRC5/my1g2NxCYfqOAoD/sAGf60B7x1N/zXQgcjD1ixtvR7u/eI2z513rsV0RRIr9vO77ANCvBwqYgsV3AtNB379nbPPvT6x8exZd4YN3gRx1HY/cr2LI0aXfelBvAyBpoFFnWCRjyrD863XzUaKhGczx2EgJ9JjaUpy++iWoktmgtayQ8qC2tzbHuWwPd/cHrPqWuzYbxBq13G4HyG1et9auFBVU934m0kbAcUHtpHba0qqdPt2VL2ZB9hlwsl+LN6eWVdTDPnDaZ+tNieVl6WH9EyDc8hdca3oaTNm5ls5f89x2/pRXhRVppooFei0HKe/bNqWZmA0/S0D33EtuW/MPWeuvTamtLH3xjeVNk2jabwTalgL+p3XLaXt130cl7zUmiqlX38Qph8DJRC5JMrdAyorv5iIECUjdBKgVENmeJWIwvma+Deby4nsSi5PiqoJRZWfgx3g2IzT2K3fEet+NVWj7jc1POqWjHRiWjoZZqVq1LY9vqZfKi1E3tVmFbPbw11bgQddPmW6hV5mwKUjNYT+x5+fvzRTWWne+KMVG8npOec3SpyzPoZeSxtCeYqWGYx2ZPQ+eiW914G+vEeHDz42FEbU1K+CSmSgDILuZoJ7HJdXaRm75FKF6dXb40vgtXKpKuMUljMFBKclYH8xwMxFmyPqRjNUvR54fMYhSWkVXDMiIsfQbUG2k2keDz2SWYehIfsMADOjTgEOdAhfNjoCL95vD7YSiP392Lxnj1WOZDiD7GRYaHxl1Y87W4NhkgISGarxV0Tq+uzt6+vzo4//D2xdnF6/Mfxc+nF+fw376bJi34ahIKDmQUX/WOpl99hSkNTHI2jJ0UzeuwHUw4wQY5QNXM8EW8FtcJ6Hs5PVp2/Cd8IDM5oMgKHy4dC6FIwV7GQ536TREEY/Ut8aCiR5LycJKMUxJRvcWk52EnEnPty1fLvmbNY3poat8fTeBkX8DJiMrbmB5wPOp9V8dmrSI3Ygy4QjGVtWZkKzZj9gDc8hmUHFWhzMa6dlMudwgfs0KTdg0fk7hlBJnVY2Ozcoyx6hZ0uhZjZlkOEYe+wV/1/svCHQehoJvdwpIiwRhvX1dMQi2ryvuLdz+dnZ+evzzrVfdVJ+yx2KuS+9rLINPOZ+zGw6/Zj0lGwdoT8RnNlYcD/JcyyxfjQ8uo6S3XPfGSOVEHWXUpCEoFOPya1Tg4ycY5vgB40llV04N/tljA7vTlh7dvTy9+gRXgdjiSyuEGscYs6BNu0vjq7pqLtKW/zegwkA5gDX5FEohFEe3CXfHVhFtvRnF1hZPNi38qfT30RKDR0cTF6ds+iLbjqXj7olOL43ICc49UPOc/sBX478DUTO+tagTdiNW3Cgd0lVBlitJJfnWcPIlH1xkDGxHSRuEJ0bH+DYlL9WnT3RZ81rYGbFp3oGyb3HSkGbXfRNRt74kTvE8ZkOyrQHXoDfnR+TkdlZnU8lvfYVAxLU5jaWCgkNrPyAts53O9avc5p6WpV8+XtMUX/URtjs6OjA+83WhhElecZwjb6olX7ALHubk4e//u4gqawKRkwG5zmTIIE3pdr3lItnd1w+Y3tPO+aMsrlOaLN4l3kcg8wc+Onn3TFfDvt/TvdzCx6FLgCCa8n0fVI07tVE/fZfwbZWvSHImAXNX6XTqJr5GNnlqVYLLnZlduCwncsonzfTK5HLfoEdo2k5WgcHulY7eSVgOxK7I2N/+9OKKpDxjxIfcKVqI1SHPlxTmXCDr4SZqmNTlqJ5VkMG3/dHcecBOJcWXsOv4mjobmz8i2+Hm2oLNGZzYQJKRTIZM2CkVmN5JLGznor8RNdbqt9qOv39ZpbCt+bYq3pbdurWnb460PLmyuzEZ5t+1BB39llfKFTxQjJ0FRkzfqp48ntZRGXl6pNeFk8tmpCaeGt4mNjt+oyZhbnsLbqQnbdbxLE+7jeK1NOM7nNryuh7odWcND3YbQE47TitR2brfhc2NlWlFpl3gbHstn3k6x5ulNK928Bz0beJYEE+6iEp+1f9PGxBvs0Zba37bX/nZ77e/aa3/XXpv2G5oDI1Bh5PjOn70/dcW3nH2L9jAn6VYDqbv9WruaDbqLw8TdRb3VSaNgPSJyXPJICVvfkLckbJA6SdpxklzdjFOC1HGSfsy+QJ1gWzGdq5OnJRoE3l2vkVrHh8LN47ojHhA610n1MUkyjTJoR8r7vPiD8NoyG/oeLb87QkOQFDePPaIS64VNm6eN1STiRZptRJz+fVjj+41Y0eH6CKzarEJK1K5+67JmP1Yl5W30XZ3R5UOX3B5Vy5i1NqDVDQlgzRB+bs4Jfm0fYwOhTUNG6FDtkdjKiYtMJw85lijxo2RE5kMMyd8dfxax2iMXAaPTmpC7mdiHzap7UthxDy07t34ybeVOcdXW1q6i46vONDIhQDolhRT+O+Ab8VLVqmcXwHgCk2iwgd3t4zbsdeit2N17891HZRGR2IYbr9nr29Qn2k2P3ZIOeuUP+h38Z86N94nKSbXDGcGe8m1KR8jfN3MaC03BBjSbp4g9MkAC17icdp40PJrqRFr813/8p2AfqeXUtG9PTjsHgm/ujthzOhLif/8vYd01xRLCMKpVM9m7P08GT42F+3T4wN5EEegCtYdimT7e+NpUlFsVFqsHC55bhxXTDkJq+w7B5J/PBZcoywyL1N+qzLK9sNgxxZwREd8II4Vh5JqVHkbWOF1T1hmUimIlJ4A9gsAZAqOwBgLjna7mc3yjoEyqelOmT1gFCxpdV+F10AeZdDsYfcbV+lRxxdPh4GkjHxXgG1kJ6J7XM67ZQ/Dga6Z/Q4QCg3Z13LpMVuep3bxJirXD2tjormAfnz49SLMDNLYOOEYeibTFNAR8XWcMNhYy2fxYXOuPsBDiA2mUmVo1g67RoDa9dLZ5XdVjvlF156akhnbtskY7ZEph+npTwzK/eATV5IDMowMwjw7IPLLG4TOzoFptMowjty9s3wAh2uBtoA7Ylr6vgl2OjO6sn+u1IL8q2mSHaFrhP9+BLLQRaYMOqh/6Cr5tK/gOClyZw35L+0DW6XLdAIP6dc41LnFeAtJEEoHyOXMT31uILSMMEfZdKfj0l7NLEuplksjDVzJ4QID0FpOnqJ4Apq022tMhqy1Ps/xpfYJZ3+2L9+/evH75C6xalERSbaW1jc3PEuDjuBSxem2XDyyW/NKXM22U6oeRdoWzUYYniLtVycbhP6dbfvIBgWWRHKgcILVWKG/CwTdaeL15+xa4LRnDCkCZLFR5kV+vygq2UHwmmBIzVYkIPlyeveJDQA6sntTn8afXP/Tdvp98VmrBQ9d9i41L6FONgXGZCa0liaY4bKpSNZ6yd+7OfnAKy+7l5U8wlriAQcdiZLbtEfdKzs9zoTIdVPE1Xmr11mBYO+KgcxnjSwRC5smg45EP5z+dvnn96vTq7JW4PH37/s2ZePfh6v2Hq1643/HofZ1fM5lvh5QXpdgtYrTxnPQgdCkAZBb0tUx6p8XNCq/4vMdfRTBJynGRLlHynNSyZSBH1lUgeUwbL3vxZBLFElfQOaCjYTTgbvN0nKDuWdIYKY5zhQ4j5JxpvJpXJ/QuaBsezhLRFfg6OfaprDAmvyrIGLlN5suTjkyJockXMH8Jk+ZCpb0BxPRgzLJHg8eG1BlKW9IUOuImesHax78xUBa1cxogaOZyFB1nNlSqli5XpEHglIDkiMjBEkWki0YRTlAUyST0RQyahbhcw9pbnN2nmMmfTPS9/wMwmQaXIs0AAA=="
},
"03_combine_model.py": {
"sha256": "b48fa330b88fb92b722b4ef7989c8670e03549adc45f3e5bd32369be52f2b002",
"n_bytes": 62205,
"gz_b64": "..."
},
"03b_metric_panel.py": {
"sha256": "37984cbbc10380ff2b80c6fb356c4c8654c05df171dbf85ac3170acae5691fc9",
"n_bytes": 28120,
"gz_b64": "H4sIAAAAAAAC/81923LcSHbge31FBhRtARQKRVLd0zulLjnYItvdNiUyJM2MHXQZBAtZRQxRAASgSFZLdPjVz7sR+wX7A/uy794/mS/xuWQCmQCKknbcEdsRLRXycvLkyZPnlidTjuOcX0eVFM+vxF/+7X+IopTjUq6SqpaljMVa1mWyEEWUyVQs81LU11L8ePzj4fdika+vkkyOl1GdrDZSVPUm3gaj0XtZ1ZW4u5bQtBSR2LuSNQATaR5peHuilNViIytRyHJcpNEW6qs0L6TY2ytlmkRXSZrUW/FyJvaDw729kYvjEqLj78QqqqUnbmVZbSpCaJlGta9gj8v8alPVqvFzkW3SVLjHh/teIN5DW55KUo329uy57u2JJBOXl4s8WyarYBut08tL4V5eUo8p/L5K88WNJ6IshmZZXstqEstFUiV5VgXr+PISYB4fHuzt+eKuTGDSmbiSQDQJXbZqYJj4Jq3FXVQhAYtNDUR2s1z8fPT2H5JsRUhCm+s8jSt/dCeT1TVQE4cEQOtNCv0iAPjTL/94cgzEPpZVssqEW0kpYOhmiYA4V7AuawEIplEJP/PMG42/6r/RGFbjtF21CijkLjaIRJ3cwlIVZZKXe7ywVZItgAdkhdQAXohKXJGkLPMSpqVW43AkgHLULcRe4RaIBp3WUVYDfKD/Ig0I//yqkuWtjJ8likblJpXEoE1BfgckjMpyK7Ic4AInIYUiRONqU4vjMyBwBnjDhxQG2jiwN4UeQuyJ04MBhNwYWAIB+WIbezSqvIcinAi0XUflVnc/tLtfr1MC8PPr02OYh1rIcbVZIygxEz+/OxbPRLRYyBR5R6Zj+WGT3EapzGpPA33eAVoRzGtghXFVSJi7hZ/u9a3di8bAfsZgTUdjVAvGdzYMuQSS1hVCWeQbwNCihe70O+gk06ioZBwqBgipTzXc/nt7kHXBJBfnZbSGVcyR3aKrPIW9XOR3IBnM6friXLwUh98F34k/TW5WHjHp2aaGzSRNPsUecVRCV6DXXVJfw9524zJJ07DeFgBKia+wyKuEdoe42jKbisOGPc6IPYCzwl8Byf/4P+KMFlxGN0zfphiXrP79Pn4TRue42QGPmZCwFbfCPfXFmSegCwij04O//Nt/BzL83/8JA8BP6D0Thwew919hPbH6jLbBu6PXJ1qSAUrLKE2vosWNWFxHMJ9nwpSVSj5AaVXnBUMB6bXOY5kGRsMQq8OGOS8vPZAjb+QdyeIIpNa6qDUlAdfDyenzyem3k9PvPZI8jdRalvmakDzYFz//Cp/RWuIWHl1exlEdTZIMxGqynhRR+WEj64kmeF3CDIAdAlUBFNRS6/jHcZ6lW1YlosgL2rN5NoJ55Jsaxq0mSxnVG8C9gacLQlZADVRYxNFBIN5Ft0n9680W6Px3OWgaUa3zHNlhJfIlrNr95eXk8nJriPqAW4R3SRbndyFPDOonyBSXl6q2yNNtXsayhJFeCNqXE+KJljDcEkileawKRoewyGlSWB3qnJqzqMbm19sqWUSpWEf3IG3Ejyc/nb09aeUJ4P5C0IYUC4AFUJ8H4hxVqFytYUvjTPbFDyKuxQ8zoekT6G0UrqIiBNC0SXGDEq4wyWajiauoSkDRVDxjFgPhIq9qEpDe1NiSl5e3SLttPKk8/Ir0178cen1SIEBggD/LRZ2X2xftnFAdZrBVGs569wtxbwhcyYMGo28DgA/SEDbHuw2otr0D9/blLchH7/LyBQ5N1GwrP0WfXkbA6eLozbG4/WGmm/KsUFZDY4SnJDJDWRcGiPOXa2OvhHcgcWB+KE8YyDm0PXnlRuUCyOFGk5Xn7d125NnkrFpl0eIa1wxImSwSWKOKjQ1mOLkIsYq34i9qf6mtCBZOdCOJiGwojUmKwQ6NkwhkmpYJsCZYPk5B4OhaD1lcwtAik3cj3tOglaN6cW0oZUAjZK0aolK9vGRDSdey8Bxr1U48j73JhCplUebxZkHmUyxhx6+TDMU9sC9sZLRb9vZA0SQgEqgNCNkSlQ+CAEkBehmUo2tzmCdmM9FRymhYXgNPgjhDwRbnIGvenL2HvbOB6QG/H1xewhaHH8/RUpPXsO3zTQnoCiU5XoikFiRejo6P3yFFxDJJtcjS4oXE5YSlARMs5DVaVLiuE2EXh4A92gNk/vVqgz9Xecar+naTsVIptvU1WkjlYrL//Cq0mhdbMR4jBrgdi1R+dQf4AkIu5MhxnNGI9lEYLje4/8MQTSegMiwK2K4kV6vRSJeVKxCcldTf/FeaXAWbOkl1KU5H/662FQ9QRPU1NNTQz+GTK0DPEp9w+VG2bUYDhrmJr/RXtlnDPMCwzQpdVKDyRgEkipiBVYuk2AZo7IJc1ChEt6s8DWEVge+MZk09zBKmCJgGiGSQZGBU1u6+DzWli4i6IfYG2nigIas8vZWuhxoErTGPIYKaWQP5FUhXiCdgb36IpuLk2/1DWqCoKGQWh+UmA3Zd+VQGwiwDGccfK1ljzUqW/E1MzW4GF5BBAdIiXF+pgjK/lRkJWPpWyPEH2PpgmqFZQWqMC8HWR/OrjlbQyhuNXqHGmInPTBI41tl/HmpVyrZCsQXeGcVyKUKFaQk9mHuBr8gjwH1mdwP2xxXTG9RNQHLhsBkqFti4wHOKiNEV1EfVOKm8APmU5lTIBaBrs12ApSGuAk8A3C9iW9dRI+8/d3yBM2UDFobuw2B8GArCc/EPrxk0wCnKMpD3MBS3deEvT5Eddk6GcIEgT8RbElyooEEKkylXSiAOKKxEogYDLxONuDOUkFNjDaG+BI/UffXuj6AiQZJn+Eclg9H50ZuT0/CnX05Oj99NRQqC8wJYcw6zuGByqwWIJUyUP3j/42fOVm9Trb91C4YA3jjYEKBtqUkBH8mvsmzAod5pPqp2mAJ/wTzX+Ish1dHmEEvXMgIDX9Jvw67sfCIaWMTWZpkg+8tYw6pukiIsUMN8F6ZJdoMts5AVXMUf4AJqbA5D9Aw3cdufcMzCTZaAsdcgnXyAiY3mYM6e/Cn88ejdSfjq7PQdUtMBNQ96BFuy4c6/0QZQP0HPw485dT09OzrWXT/ygLY/5kxFA7FXTwNgi3akPgwYmIBoBHot1gXWM1b+6IEYcPxf9594AvDOQS+Ia5nCOgP3GrsTtHSNYRzvv3xQFCusOVvTz10sV2hSLoj7fVQVc0+MXxpFS6BJPW9k0Cu2FW2zGSxGw5xURvUWfEDwmqOFNCs36BWCVL+fYMRKkCnX2Ml7IMoai518HjDVEE4FiiNng3fChi6aQreGL7TG5gQPLWAyf9gUftHBLk5uk5jobNq4iDRZZ4F4g79WyQK14xUsT6CnzvoBGBOIduEQKYFrSfyB5psxpdziwtFgnTkLs6hC7UetXop9U74xixP8ptMUG/ptxQpKGtAr4M7qEAAbDdC4ho6V2Y6cHCyH5tAatI0NFI1z7HNoduJdAxVqjF6v2xC9nuGxqGbnaLrfwHi64+CIXR/A7N6rs6jCPiS0B2+YsBxwLe0ejV/Z69TW6A4PSk+D6pKZLFdbsqDdiCYyBXsqwBhIGW19sVIo++SEKF3Du4q2Wtu22WQDLozgYQQOA36PcP9+crOarJWjR5SUHGwU7ppcwGZ7naAwNfwkdrYABpj6MDHwEVzEzBcn77SLRcFYdItgn1ULsLHInMyghb0bwIaf4QQUdPxV0VQUJXzwhLeFnEEFzfh33+IS8wqCw2ZuBWiisTHAMGI9ID6M7KklaNZIyQ333qb/1v5kBqAV9kWzsPTdWlqPhy7u/W0g/kRwRHUN/tQNyPA8jn3Y3g1Ej2IrwJxo4yuHsjG6MrSWgMXAZXPvwbb+FSNs9Lciyp1qwNhyWbKEYuCXacOy2CrD8kx8Iw7RdzsQMq0kfI/FQdvpB/Ec3bE7DEoQZzdItrDUMtwHi7zYuh7iw7/MRXItw99cqHulKaDjnd8ZxNhkZPztgrF9DIZebkVLZWtVbm0v76OLb6Gh/wMZMu3pO1+oyIC1X4cBYESmjeyo3d7RoqhYG+46N8J9FFvGgF1WYShRR/9oV+eZ1NN9Wukwn9vEtm7ADgedBc31Xn9Lq1RxrCbEkK02hHyh7B1fkGFDoSJUrJ4o0k1F0SwObIHGg24ZMw9+FmMiHjK+FVrjgDgqWoqQ5HkKn4syr6rx4loubsDYAONbySD3FqU/aPEQjXNQwLGnvHaQFKgXYEyrSgcmdZmoJpGIVqWUGGezpVCthJBio3pAYFC7e7vd/a52W7vddlc7cDEate/Ah8PF4A10bSq0ZsGwVlqH1UvN+91v7NkpAjTNZC4Y5Dmts7QVq/uy1aq/MlautFJQuP84LNVaG9HgrDu3SteTiEdaXMwRYV7FbvmjwCOt/TugoiFQD1p0MY1Afh32BBVQmRcVuGqLaqivB6ACnb7yQhsDc/Vp6nMCEisWipPl0lXHQbitufR6W+S1q6vvMdqqP7aVFtioNaFUliUGP6TLdubMSVYZePEOyjOKxjUlhuy95XHursFRcwGXl0h8Gn8CqPm4msqS5IbLTZpCu6C6jvBAhRZ7mEWBhrEi4suZScVYZiBhZgLAXBxM5+IZtLuYjg/IBjsM9lsd89dMjHAm+OYEaWiao3tLlWNx24xNtWrKBIf9IJZQ3Xg8iMMmZB+VdbKMFuA6UQfqe0u9ZrzqhvXKZntk1po2Kler/XCrVDHgjwFT+CuplkmWACluPfE3gjTYVYUfL3lAT/GE3lC7AUQmgAgBRCYAxRdY5N76YkzQfTWIyRDUIoIWEbdgKBb91AlFhc4XClYO/I7JnuycPrjtEYat2wzKAjge2iaGWYB9aX7M0frnD1pkdmCrKYPT0i6YcmzUcuGBQrNa2n/humVEmxWxglFucSXwyIFRTfM7sw4QaCuvQQk2lZ9ZGhhTdaKTEJ6FWtWYrATAYq4moo9E+q0In7+hkXXj/omIWrrBw1nBh7MuOLYbwBXlUQ5OeXuapNdaka1K0AEQe0y61jtVBKfq2x3VFBzseTlVoiTpymlNJZ4K44Z9ACICVx7yMCl6ZKfuRHoG9FJh1ff1kHZa3wabAo853I+tLoUfti6ln76pMvGXoTPXRV+FdVWolgl+T13qzwEYWove+oYejB4syxrVGRu46pg3zORdY+Sy4WBaFa1VO/f51BYD07DmU1h9jmXuVsi7oj71pkjlRREHx1Ed/YRj+p1mthGrc0Uaa/bnd8d8sjrBPIzJ63NgZ2UZakNujIacaSbSWUHjGcEqo186EKfyFD8Sr0EbtMtdMxRz4ajKRqtbAqZhQOqjY07YrX88ixB4OBTBTH0AjwQOE7016sqoq0Mtpu7N4ntVuDULt6qwMgsbKRcZhREUUimGW31O7ql8NtSV682BWFcCczErhWDryPvZ+3IjmyLuQGXKRkXrR/vuqwo0gcvAvceGw+IL6jrXtc0nN1KfTLssJruM2gEjcAueDx53Kteqw2AYM9aSKUpT1chg97ZB9bkG0ecbFOXn2hhmgKHQ2UBCL4UCI2WUraSL9GGrvjWBZEL6pS6p9mKleFMZZ1hL5zOZvYmtrYvZVEm2kU1hBKbT1b4yK9Q6rNCcxm8kuzUMmnk9r7m6iPanV/uwcPftz237E/ZeI9j9rhnQ2uKwjAGfwoHsbfbHFOdlOjbr5hhgbgvl9UX7tUtgGRIb4aiPuSm5L/QhwkOLW7Nuz2bYwBLk826ryG7FQn1urtT6QsnxOa2wvUI3YtZtYNU/AQM3meNhHOlolSyi4mpRzftdJM8OUAqj38xmd+v8Wo0s0NiGhFulFw9M6gPYbjdzqx3tJr1WLa6eHZoxGxkuMA8y4GTY3ZlMSsUN0YloFTGxrGa9Vk/Ab2CKLYBUJR8aKArsIFJkEKkHL+pMX4/uDbRkmTBAhijSJN6f3kTzQXqwuaPCOKDJDGXq4nZBSZlu1lk1MxRKu1c+e0ymRqBDYoqHwOLIuOpLMJ8oM1SOB8NTFEk7g1RqMVEyMXSLSvrwYunckwoPPyLEhzDjqEG3mE4twYoO0WNuAvgcMBne8l0IhYzKKs/CstP7oeldKD8ozxZgB2ZoCxLi7fJ+GGqR37YN8puOQ1N0HZoPhnBRwje/IWPW2AfryPINcPLaiygu8ht0dj/g3+DjceSWAra9IBLReSi6pJcG/fnnFH5rjQAawFPuvjjo1H6waztr2ozF9pgieckwfYXzxb6xYR5jg+zzbACE6q/+YytfPijWT6KVCqihsC4w+0KZ5+RmK7nvNTY6ttBmutEi6nAftNaBeq0nVYdUZq4q8QDs3h5ox4cGF+18qO1IgtZnUYqxMkLN83Y1j7idljp6/zv26Yga3ScIv8mJ9JnOLtOJZXSW9Kybv139JufSTTJbSPajGy+nluQkQWUWNG7IW93TzKkjICFwPx3oX162qXNohNUhZYWEZDiFYLnd+CIIgvbAbLANRsyPD37vAeepvDyXkyZ8nXvvoRV4o+PkfHqsk68R5NNKrMHO004TG8aUPsRoAWROk6bEHTQtwT+hNuTl+ywp2kxqz46Co9iJl+rQBvONZM2ugBuXeWHY/jFoP2UO8vQQBWfOyqrOQ+ROLHJ3NGygEJ1u5Naxw3ttL2wACh5gUqaZS+mFTjtJ59GYMY1hNDYB+W0dksKqUx6OJvMMmyaVSrSMEdnGfMc1rigKGqT54kL3mXfQClZlvimuti4YiMtUKWydcM9fLS0wVaGsmd7DBOxCx1NAe8XUHBg/ReguQxLV3QYWtw1w7PA2SjeyegTbAaysObRC/nNTB4UCEoJcO1jeZ+JAiS1AEJwFivooxJrJfQkRvZ2TtphvkCYXrYZSWAQrWbsuiX7gHNhJGw7hN3F17ctlXIvu2K9JgaysUJ0Tw5nz9u0dwHDm7U78wpUYnka7RLu3iMnT0L4NTvh4MhjPnLUsVxKxcB4VB0rDxDrzMOmHn+IvlcacQY0ZvmN1f43DzILzOJvrQ9PhjGozj1pnV7eZirA7H93MsH5XeIkEFs/KQmsNnXRThwWvjtru80H+NoWOd4FA5wHj6HoWsDgbBmZCeBxA/SgAkm07AWRSiTeuT6oscr2u/X6V56mLLYMoQ+H4mdBCtblqMMJevthBmOaL5e/cEERpyrrkHeg/YH4AuWP3W3QOqD6A7eryOnmtTGc/SyUH7NYZHPnC8eiXHQJBvAZotGa62GShtkiCNQoUBEgfHZQbXOPstxtKqbdmqLoFaa4SsYAeepBuQ5v9apOAf8nRVvKcXYpC2dtd3++sOsWj3y64fFTX6u5Ea/nylvdbW5GtOl8HzE1LlbI52V5VVt0f1U0IjsDkg8Yu3ZBgaFi3644E5ltnsSKnulCBNxzqXMBnlIr62rRGe3cc19G9AD9IoB+ECOR49WbAmONbVUYSTmyGoFkUtt9BVOFiu05+hXd8lLfIwo40gdtQMs9mZujhOr+bOalcmn0GtUBb3bPXm7PGwUUQrrxfpJsKSj1M4dYWs09Cm+/pGZeUG3FOedWpb4n1JkM4AJd8XZnbCdbJEIe4E2AH7gf7xo65YJBIPWjdiNuebicT7qcIDG6yb8jHB88d+liHqvp+DS15eyNHhddvk4iPWvUV1tOD9gD1CV6iOZnat20w3lXVOYa7oor98eeHOHNKheH7dZQKC8oxvxORqNbwC7kpB+EgFeA6x1xASn9XWTQaEsKOVoAq2GuxymYDxYsJPeVdouMQv/vWXEaYO/Pm2dvjk7fh25Pzt2fh0fuzU6DhAd+OJNIfKI60NovTXwfYYgdM/oP/F/pz9BPj0LR/jFALWHNQrIMtOMywIAQ4MJ5Fd9N/MJt6rTPxmOlBKfPhOoEZZCu1X1SAqNHtbYhPa2ozcgSMRCEN9lyJkyjWruMl4S3m3JgYTxsaWILYghABJ0DD7sLt7kD8DD3IaGho/MOsB8LWwnxngFynhgqG5KDT0T6FuiAUYekqnAq8uP8K3TShHiwN5iPNfpvcfHouAG+Z/EbBjk0WIvSuWe0PqU5UbyVUqSsjKcVtO6q3vY6SPh7WxXjJJqO8QuNa9JSA/8f/FosywYvDFPbRjzj0Lj7jzZQNX2JrLPO1Toqn+0jKRxvMlI+XfGeJdpM7tKEw6KrENE9pHoAZi/uFKjQdmtK5qSGfiFdRlmeUD6Q08OQ5ehxKItNjFuqOM0vTpxW9GKFMAg9la63fqnhaKaCnB5OzA6HuHSkhXw3cxAoKvM0s76NFnW6F2z6qYBBNgXwNhrccv4rKNAeYsO0qqn3BB0TYFK9yoj4gxMaYZFWC+U9HhfK+ABXPT0sEPbqqn5YD2jDMbmu+cTgf8SwbMF/lYWqMLhx2dPVIcwPbXl2jNzxt2WCG9qi5xmf2bZmiI8W76XCqI/51YUb26WUKfdzL4fI2aUuFz6kNn+5yat2H0tQ+sMR4UpWk3NIX33+Hmmao5vA7jv3jF4fjh88A1NUrEvmHMBa7chq/lzN9S4Evaq2T5lbWnJ9KQQxftvcl2mZ4W6tJn1lc55UEUyzkHT4jgQPUr7nA1VQm8eTbSM2sr89E87TbqxZrpn/4pgCbGb9V+hYwekgKlC/FAaYaSWNbhXhrqotrf6gvwnCHbLVRU8qoTfFiOl5YF/Pmegp50awf+OQu9nupO66tPp1HInCdFJCbpAFCAMG202O2Vw1JiDpg/cX5OqT87YUstGXfv3VkXmycinZFulcapyYhjDsz7cDTIWSsi49ti7bMECnGrci2ZVs2N65KduqhpAen6DYqCIK6VznVvNT2ojuWbR/6nJt3Ltu6psgc1Vz2KTLGwKVMZVip1d8PDs3rR53rmlPiGn/gwuaUWMHo2V7fZKuplak6ehlkSoh5XnPDsz3a0u3Nixrd259Te+P73Yug00ZwmldCpyiDzCtWSHW9oclepHmqDW7drEIribTwF5pJfNhNBTnGSZJySnewu1uer4PP3oAR9NVhCTSe+q/KoKhlG2LnwwXqOQL6bT1c0JhQKM/Q7zbufat0yqIaTnYzti7mnChv2NPpExfu6YFve5cenXPNddLkDrj2zu+ARrhnBySKl+GvGuAXp3ShP5rirelWNqM9g5OcWoH4nBsZQgfbIdLdVAR0tFqLWgl+22a24Hjd/hfGVW9ElfDrN7LufGO7vN/OyIYq9VV3I/7HrNfPiOHyIMmWuesQS31TTb6pUIjMvgm+XWJYZAbfIFjxLxZdWLP6zAGaIrUiJk3D0k79MhJT88ehUh9LyFOJEq06QYUfXRtIgPHsavz74uKvuW4/F88G8L1YECMtkG/My/64EAuVczc0TfevQcXbgcuQ4JvPm/RllFbB+gb+dPl1iEqlb1LkKMxvDGMahEoYLzXlTOeHa4y3Npz2IQSHBaTVEGxl+OWq8fFBiq70ossdGFLnSAzjq8RXc0FfXZ9vRuL3Fyz1RL60oXBoAEvbPGIATc203UZKPWox9QcP+6qYjGfivv4GUBEH8yIyS0PQwEsM5Vnwrrah4gMA+9Faf9p6UwpnydgFD4mHpKCQHr0rWZRoAX/3i63VXfPoMPC8k69nSNpQy9emwYMxe72AjDzmCICEd/MS31OaOfwWR+U0d6NpH+3iK1SF4C+SxgSr/b5unVOsCmLw3/DQiLiMGRAGOeSYIB626s0BVkK0SesZRTyeCeefM+BXmcF8YIFmzqZejv+bMnx3IWPo4h0o6RbhOub1Ig3j7RpIWdiqpeqrT1a7oL7AriETBb6mRnL8V28HnZLYiNtd7MJKGoxMutfdnqM7T8Rr8zlSLQIGni2ll1gMteQYv5fOmMNqU7G397EVBA/4bGD/aT7vhfXKHr49Cv3+8u//6yOmX+3tDUGGaUoKtWMYRTelrs2YMFMaMV8KE4cdODtPnohjfPIKHxJVkV1+QrX/gqqB7d8OgJvrG3kaC4p3GafTSHdtPzh7e2/yANB8kwsVNlBywtOBqGhxLSuLRDBh4QwLjaVDpJgah1PP6fVWfqY1qYA81jOuMDQub3UHCGmC7oDtGGc37ROesNfwzc2JugkFX/g050RlBFT4i5+u5BSoyU7onXtPnlCHGooewsVHnNR15Am96NQMWf9+3+MXy3YBR0PgC9YzUPsbAzW2pRqC+YOyE7BEQ7lEE6d7xmutLPIqsGJ58dQQzU/nD8j5WGqbFFCxtze1VtnZrRZglQGC0fjpfAoG5INwG3tNXGKT5hPgX/qPg6TVpUs3DdL4haB/t3oAw6ItL2i81YOnyMWi5Bk+A4TW0hMdXKe3brCkZ8A6nzhI96lZ3k+ixf2TaIbXvwv4waYn/DDJ9IlE3icOvnziEwjV6VB8Ghh4PB6b/0+7f0ybP9pPp/VrNCOwlhtiBYsNbEUM0x7giE/DDKHKzTWEImfU54TuYolP/bVSZUzDtmwI3ABjQWeOaVi1iOvTuWdV2/ENVdsbBa0yaEyr1YVghyS4VjFaRtdRlV4zgkbiB3VEN8CJzSOn2LvNcEUBJRc1vs4UgyDBm2o6CeFv+1zTK2lVEd+c1qDbN1VFs6FIJ1GUWa70IdekoxidHnx6FEXQ69Mgtlu4nDhlK01Md5f4JMSVrO+kzGwBx0cKC+kFXzAxZ889Ak3x6t0f9evPkbhs7f5L9dYa327A11/pWUT1uiM95vSHN388Ov3l+Oj9yTG+r3t+eiLO/vD+/A/vA29PbSRlP6EtF/w5TzKX1k2bd79R4jPq0Zofz/lNDvw4SImnTi6+hshxqR1JdLBgFN2mRwxdtOKiuJodWNYl1NKreZtagjZ5d3J68uq92BM/vT17jWYBJvTQ2Zb79COO9/AUxHGwlPXiOl66+h0VQorvGQJDZb54P3R2aNwbYfyqDxTx1mEr/E8hoI+JwDQu8sV1uK5cVOx4p/9gf38/2BdH70SNt4j4YQZ8ZqSBQbh/fP8g/vTzydsTgcclIEXwDAdGe3p+evRPJ2+f8im0+PGfjLEAoJU4M0AiQFlNXyUItxFGfHBQqqghPd/JEVoiAuxLtSTLlTozUk9HuvowCUUZVLWvTKqazjuRlHlK9npFl8+UF4Z7B4VWs4Vcil0hSp4ZGaB8G349kho4ak85vuhF7sDHuIIu2uhHhypEHsB41WP+2GjUC0TNHOCp7w+9fg3/KwTNP15g/UsFnwh/DFN9MmYG3w7PzIxDDA+nreQG94CiIKBJ+T4LGGy8VKPh+BleHylAdODbOvVUYNyM+gt3UzXPtGKyVikp/Q28FWMsPA1tnWT7gVFF/u5bsNN2yGaSwO6++bgoZqw7u6N1HynKX1A4nJ1bcM4iXJjpwGqpCb0QxoSch97FnH0Vr6F7pYoJdjFSex8a96vz+AvfzsjIL23BN+tkvNUSYVLRT0kq3+T1T5h0dIL/LgFILZUK0uTJXeOJ1RI04scWIJitQHl+f/ewwQWf3iVVg0Zm+/aK8QzrrnXCLaxWyMCR0wFmppRuUWiJinPvEVATpG02LL+VAKA8dAfmpDLuPz54XMTP8oaqOdQceEao5j1J3J5kB3wm4umux92fothvPRV+PGo2IPYH9iK9zBoryqAVO/sGRC33gp+AH9ooVO35A9fj1Sn5Vy2OPm7orQ8mDvGtaHqno+ZxKYJGvwZySKykBZ0+gMrNSNPVyZBrdZ1s97MLvoECR3/6FDMyVUHktE+I+0JdQDOfnQcSTpCg9+oBBPJcXx8dU0hflLsi+4TpxdAttbnfVtpX4oZq1OXmnQOoG3j8rKN9a68FZzVqb+nNv3LVSQL315yzu/C0ezhB2W/Xb3g9ON3Hyu7lR4pAHWBi2yeczieg9POVuK06x2QDtIfOFwOZcnO/X/N42t7cM4XnLrBdCXpEL4jCLFh8ssrfkcVM4Q4ETu+ZNgLWRuMrl4lNjN4y2WFOdRCnDm3VSRydBWjbw9enr/yX/UAR8lWWV/TP0VRJLMHHECpnGeUpbN/mTqJKu6U7d01CLTGmOhoxz5w+Om1+A552fqRn8fEWJ/V4CDCk7HreZ3JDHMqk2glE5VmBA+dZ2HzBcUzT9PEDmdAg0NDhjHE2g0+aNG3psLKL7/9fIfr+/lXxcZCk5Ev7+AtDu6Zz+xK27+FSKSN2mPtr+CWnPF8fUR+QOBR3fHv0GpAODpbi9Y9OxwC0XLh95YOsMcPKdjfo0pz+VwGCo3K1wccizvGrdPE51jIpkPVnXTv8M/+ImDJSoiKIYlACCq7r8D9jgM7EdZ4sJL5LwP+kAZmFmzSlu25qISltYhccNkTx9THGjzLZwTChd8LxsWs8KiKJ3oQFXH4zTrQmtA4d4rM0SIgiIELgQJU+6NzhkNE/x0C0qwL6jee/Oi2VHvrhWdixBu0G+tyRJoEuIojokJIjw5ByqsIQFysMnemoFczvtkDp9cl9Uru8lN7oPwHSwa802G0AAA=="
},
"03d_warmup_phenotype.py": {
"sha256": "17d6cb5976a1265d6f0559959ef940df82bef72b4e1e252aa4d92e86af7ef10c",
"n_bytes": 59046,
"gz_b64": "..."
},
"config.yaml": {
"sha256": "d1cf93f822cfbe16d51946985280ba074833890c3952b32473fbf0ae7c38a8d3",
"n_bytes": 27652,
"gz_b64": "H4sIAAAAAAAC/61963LbSJbmfz1FhhwzTdoEb5J8odqOkCW6rCla1EpyVVdUuFAgCVJogwALACXT663YX/sAGxOxj7D/9xFm/+9DzJPsd87JTCRAyuWpaVe3ROJy8nYu37lk6pF6ffa6/0ytolUYR0mopmkyjxbrLCiiNGnvPVIncaxWQXGbt1Rxm4X5bRrPchUkM5VHn8NcxdFdqG7DLFSNJFXLYBFNVbJeTsIsV1ECerOwSXReR8ksShYqX4XTgbo5uf6+vZy11Vk4jXI0lQ9UkhZh3pmZC3R7b2+VpX8Pp8VA+rmXh+FsoPrd/rNur/tUOf8eqXn0KUSv8ISapxm6uFF5kU5vg7xAn/IiXIEcjWSwp1QW3A/ULCiCDj6p+r9HKt8sMR8flfdKvb9+rSazCWZpFmV4NUqKMIuW+nX97eFXzQPy8irIfluHRfXljr6KlzH/d2FWYAyX+to0mN6GeDNdF6t1gXnSH3CJVyosL3X0BdziydRzurc3j+KQhz1NlxMss19kwfQjlmOwdaU9ze+cB0FtHVOrtQv6sVUcbLDSA/NBX14ES2qdf1We9OnXwP1SvT0NshC/8vV0GuZ5aAlv3XAasl3382C5isNB7Sq45UDf2vnWfB3Hg20OoMveEuyr8nSdoVXVmESLRZiByae3mIrmHj3o7WhNN7Pz5uHXbh7xzT1qlVZrFs4DTPZASe9xRY+Q36+8Pdg5FfwcROguTIJkipl5f/HDyej87ORmeKauT95djoZq/P7m8v0NP5mtE5/J5BE6oOZBnIfulDTCJjfj0X0VrGdRoaJcZipN4g2xJE3mH3ePHtvq3Jv3o9GufhTZOtyDCvHwT51wq41LiHWoek0SKDxZkLqh+3vcq4GZKmoLuiz0bz8PVK9bXeEkXUZJECvTK0VPqoaMa07LO4vygvrmZ0UKHum2u0c1LqH+o5Mz7y73shBCAjmE8JoXVRbGbYWXw4y/auKzJotYms38T+hfssAofwb1lur18euDJr4JMujaRpyib8GnKG9WOifvb6rvHx20Dz6o6vvQ2Vmxg8AqzSNS87mfzn1WRWEOZvv5x6uWuhm21HjUUmf0/9cf9ojcKbR+NKM5mmURzEKxWUEq0PxHjDjG9UxNNqzYTDNThS6GmXcXZBFbFCxXlOUwCOi+oeYzNZ+pofU346ubn/yfTq7O/LOT67foytur4dA/HV8M/bOr8xE6dP0Wz/jXb9/f3IyG1LcVlGiQbRxK4KYqHZ6QOFwEU5gFLHYcejKKVTT9eKzy9QosFGIlaQzzYBnFEbRMAHL8GKbxrNenWQPP+EFRhMtVkfswNNCIqzDAvPUrjGEe6YgCE5NkJi7F4q0TrEmu9uX1cLa/izYZkoE6qBsn2FAlEnA0UK9eHqh0kofZHTpfa7atrsJgRu3cjG9ORiqYZmmew5KbNeSRbmRwLTIXCvMg92iRNDmfu+uvE4iWCiGwhR/N3MEmar/ehX31UqVAFPa6fa+Rw6ItAxh4yGR5+SW+6Hdpnplz/HkcrXwLPLZE8JH6Mof0Nk5gC7O8aEKpdttHX1QwgVpR97fR9NbOOrGezHtOE5KF3AI61rjH11vIikdoZXasXh/yQgeffGP1Jhs2p72u+Vd24KTXU4s13t62pWq5ziFrpCUTdXXyTjX++ntPffe6aTW5n6X3MrlE/PD50bMaqLGzNw1WROWrpk01+r/0u81jTRwMDmyTB1i1zR5L8Gh8+j00f16sZxuI/2odi1Q2jCpQiyxdr5rMrcQkmskU8WFbvV0vsdIGoamz3gFTbZymCZ5HZ556va7XfdEcQGeo8cXop7b68YpMxPBvp6P3Z2j63//7v/KlxKORMDhK0DTUBsQggbZKE6aZzud0A4vIzBK28CgBQZrVSVjch2Hi5RuAuiWYNsqZUip8HU4DEvEFVHtbnWhhZ6LVUWJmiiifQ8wNHNXD7anGfh7GaA1jGNbe2sf0Auiai0wXmm8JngIGXmUhzMAC2l84y86hd6BG797RVNAlFmDM8RILQLx4n0Vg/IT1Ii+OXy4OzaWsHtu/84S5FG+BMbhbDixvXDwjzFeD4XE4Zw7Ms2mn2/NZNbdXm6YYTKVJGcXjFxFYDJqIhO3FC4fR+wNSUEZGf6fnVLAALltCghXJoZ4RTXAZzqIg8WeFnxOtqvklggcDa4PZ/HhzsuXq//0rDDyefvu5WRLDbBUbP1+B4Aai9vPBczJ3h32xliB2OFCHXW8DDwDiES1uC48dGumEohfVBIJGs7Rh+5sAg/jhp2m8Jm72V+RlkPWt97LX71z0oGaLW6zeP/Fr6t//x/9U/OqMXaYiS4nXVgAcWPUi3VPf9u+RuhhfeNc3V+9Pb95fQT8vo5z4NQHGZflRDcgwVs+w21/ybydNHRWuZJYDipoWcOxitY9ZIKqwOVArRbhIsw1Yr+Sxs9I0kHQYZWA64eXTdBVCcQP4MKgh42GY6ZG6uSXYI0aNlChMc+yFc1pA9ebk3TlG9VKdjEbKGNbbqlpx9EibCV5/D6svtv/a/69AJ2evW8AlgCdAKaPX/00kz3YOshXNwf6TOJ1+FFeVHogSXq4Z0TRDE6RaGvkB9+vbppeefPmKzFe2USXw0LMVFNpz4RZIVGb+hCQn2/wpjFMDYHUNntdVeL5bh6scmnNVhV8OFZmPs9eM5tFPQX505cmPV/y9Be3tXHtyNnoyHtk7ghfHow8Wrp+KSVRzsAO5pga5G4aRAIOoyL6rypy4gtbO25qt2VYjrDHh7ZAUPqjZ8AGs01PvrPf8mCICO2MLpksDca7hW07gow/KrlYNsXGx9d2OERdFqAg69i4kXpb4CUFLwJpj22MerYOn+HFAqeUqyqIpZLLSVnn5i+ashPTiF5fNIHSQD+fRlxh7QTaLAFwjmcfAVC3njabTPAH9gDQ08GbZQM4YFmhORIeMVUDUPKcP+k0ZdDSfA30qVoxLirc4tJQ8FxSmUdVj4wa2A1aKPodmDpwxUeNnPZgw+xg4BR2Y+5/Vj+c3b88vgB3rFFoWdhkJoJEWL7qQt5yDWrtaooaeD9Q+6zUW1L/kABY5w1YorRJ4QZF9hN2hsNJ9hKEmJXpmWuR+oIsw5uwpwhfFegyc12Cvcu0rspfijg4Kkp+n5XzWtCwFcAeAXETTXPNN+AlaDfq0NKZb2n6HIZ1ZO3oM4LjC4s0hUOglx0BWvjCMP4fhTsH4vfZRxYOB5CzYtDMjEIH72xCKGmhdvVLylnpc7ZuJj618gs/UCHe3itlpUYIMboc8mpLvqKTLeAOsQw5ryGo0Ee5qQLjCe1YHpJoqvjk1Qo3VGoIYsc0BvAYI17ifXjbDynk9ZTh/1VzcILOY6X4ZPiJkw2Cou8sOkG73xN+w6oswXWAREcCs2uIzu9Y5bD0rrH9WUxY7WXF92Uzhs363Dkxe9W/RUWrijgA0yxs1m4T3RiQMFdtYRPEJwfzSDF8IfbwM9KOxUrWZmKbPyLD3ygTkxKfSb7gz3ZjDnk7goBiBW8TpJIibZWP3QZb4ORyusMYaj9SPJ1cX5xffQbPoh2cKHlJOWIsEkNeI36Q5tSLKjzTEKAA9eSzv2vEphaoUO3cCSAXmxcxnq+kndWeb5WCX0rHKRjsTCfqXV7VhpX9lN7hbah4HCyPdjq6Qu6QyII99DUYf8dPWt9ezQa4QN0avK34dtO6iuU9Pu07zkQNqNa0fzt80oI8oxJAGpKJa4kEbL6CpDEtHZddJ+WPgcTTd6J7TFX+dh7TW6WqFAcyzdOmbeMzHEMYmyH1tNIkOv5EruiNSMk3j9TJRTwgzktjD/74Y3yitZ2sOlfGljCqFT3VMNj8O6G5g72fpBK43w+hpKtEncQG17Gu3l9XBmtQwGA56DCOLSmaPsKD3pHICLGRSHNsb4Sf0J2d0wq68Q9PIK6Z8GwPhicUaLgn8t9gAoYMdQOjAAUIdDnB+Mx56Fxa36SyN0wW0a75nsiKTjUDBd2qSReH8mL9JrmHBGN4IrZreBuTjJTN+hJjj3/4PmCujMNIUbiokLCWgRTJIAc+MQmZ4YxthqTN4UxzLjusQS2ZA7UZYfLMjb0CPhQMDAbaTNK4h1ca5UVpmV0gxARp4gjCNaqAq7F+P6gp7gSXSnBi3YWX5iVFMzZaaBB9l8FvUffN8pRlzkdsrYzuMtvk11XAYF9Y64kgY3ekzVgPq+m0dkptaU9G8SsbDYVxmHm7QzaZVTO12u0Ix+i1jf7rXregIPKZOLs7U+X+5qhFozNLpmtidcm01tmCzTNNfQ3WVzp7KCjU1Z8J0AD3YDBCYmpHrDkObYbHTpScviNaFCuQ0mKO4BtWvSoPKF5RMiRLmhpb6fnh5o06uvfNrGg+HPaE/6AEW6Hi59NMVDDv0PSW3fo4n80WO9eaf0wUx1n0Yxx/Y/meRMAGHE4+lefV4GodB8pgjoPdRkmuq5A4XNEqYc30pC5fxVq7FjPndiMOHDXro5Rt6xjCEP6U1yCQ7Un+N7mWhZimSZeEQ8QmhLMKMM6LBBF5wwWkbLf4+D4JGTD3jCffzGN4+xk2X1gkkPgsJsM3Q8xaJpq8RQThp6TWSZMIUKp5cxXDiu3HtaqCcpQ2AVUfJx6Nrxe3tsqfW/j2RkdmsgjRSBOs+wW/oP/BfvCtmPnyt6CmB+I1uS/0QZCSVDRkjhY+XWDY/alwP/eiXfrPJNi6O9EwBKaS1cHS/9gQN1WZFRU4fqb86o1jnwSQOZZQ5ASrnbVqfKUeQVJdQR8oY1PoxtWl0b4HHZun9gJ0szOeSkAxFq81XA/CKQNbTnZqIzADMDs8njIF3H1L8jJ0TgDtneXja+X1Ll0BLOp/nIUewqflPIvuVieh1qzfKUVAImSIT9cuVxROzKCEMSnioHRkP9nttzEXzJPfVn0bGBQSIPyKLvUmKW9/p39OuvSoLAxHwuu2nLYWfBy3FuTX96ekH+6zN1Zgc3AFl8Mr7MGRwsawD0bc3xIbo1KZtmYsbDvv0X7U7vk4/9o4sqHgnfuMqSBwsMWlygP3yauhdDb87v74ZXg3P4Fv2ey0bVut1LeI4M8Dp9fDN+GrIJRNCULL8ihIjNqdJ9R1vT66+h9IF0Lhcw3Ej4FcQKIRrBriQqQBUJyGFuYRXtHcLctN1mLusJFzo8j6WFVPkxsyP4A4W0HiPFDHoWmLngK9FSxP2xFxa2EQhz7a6KcGTcHLe0bF2CYe+Of/b8OwYVBlg8nin0OYqW0MudYT++uTd0JKtgaMndZHlMP4TqjZZMRGQbjCKaT+sOxSPFhNJcUS3sgYdvD7H+x2oXPz4pd9SP3Y+LprUYZux5cc4vdtSecqfvHwVTMM6KWjziKTkLgoUcHlI1n67UqfZ3uN5+BM4DcTgb3KBCXvXx6p7gJaLkOIDuYLXna6nt+Gs9MMcxqCA2XrJ0Ueg28ur8/GV1jERgVLtvYpr27JRwipwy0OoVQqVcAP0j3BQm1fRKIkn1plcU31IxbWcBhmQHqV0yiiOmqwLdTZWwUxMpwnfcZu66xIrHfW+CiSd0dmSgAavW1P9zG4EeXEabkpcddSvULxdxrMHKL59Nzor19vL10tDe6DeXp+BI4MpGJtAeRh74W/rSApPRgfVFvLZQ32+hfR4Evuqd18oHVYocXO7KTk9saSkS0FMIaYK1aMKVQku5Luo6qyCJNPtfOp5fDpQACqrnLxqHUzJneCQ3LI8xtks2F2DGyV82cjh4iSV1dLUn1X6uFxV130WqUu4bWCplI3uhJxmj1BjtjWTYGz1Si0d3eDff1xYcdFeUCkxpER1L1t/7AJtBUcdZ8WwtW6iytnj3kOOlwZtJuhAMgxMiBXQfCyrOO7rICgvfY2Kfp9vhZkBcXgJrEkht8+7cy6ScuTAxIvuP1EMqBJKc3RfR2nVrxrX58ei8ltlpGASgnFCbeZyMw8SgrnNsZzswh/V23fkoVTkjWC1ytJP6mm72zs8UptZh/lYhg0LDmKYiv5WYtEViW1q/Xbv+bM+U/tF6k5qDAKSR5Vw59d4rqRPtsQOdUrVFTzYXq9d9SnpllrdbnJOLkAWIlHoGiq4SWfdYwpXHdSGbxrABPQO2s/69eFLM5YCBaqcEWt7xbCtd3i4xQ49PMx2lO8uKaqtG3vRft7byUGLLLiLJFRDw6ryX75MUyyYYGmfA9IUei3fvqaXP3/ceN+lQDFKHoTZU59am3JmIFnJTCwvq6mS8iqNN9rDOqj2q0aZHqSIPvrIz+/VFrgzzhdJML1VYRJmi42iyGtbDa8xG0E2hU5oBB1ABnXZYOyAy8NT9VjdqcadYmQRCLYAqiDKw9PGv+C5zpKe7B0dtQ8fD69/OYI7dNCVz4f4fHjQPqDPB7Ash0/lcx+fey/aR/iMTwftp5KyPU2hmqNpxGF2AJ9McnTwSzlsCSWF5xvURruNn13BrSGMwZrTxhyZT1ZtmgdcJKLh1J+CKkA2dxDInPqGX9StFneoxV1pUTc+VOAGusngrqFhY7BIUqoKZv+VwR9Mpg6fmBfcIAphU/hyOiICbd11AEE9NFOPr3DRku0OaW9Gm+Q+mVh8A/TJW9Ma3GLk2/V8HpPvnVNuhHy4iIuZAWTd6OF/2Of7xzt95eCMAQUs9DT8d7IODR4JL4YdRN9U+XpuUqFlK9QEfDcfzljY9rm6zQPcBFYXkaMEngHNfxGt1xHNEyyA5eE7SJRZqjhV3gko2LLigklT2mJaZupM3DcvWA/OBMhLv2DBxaTLKMvSLFd/5A64UWzqGEdq6AKlMGyw/T8RiLCO4wXVr1C1W5iQKBrXcfqQ69j/ZtdRu0vTnd4jhnMXhfcP+pGgKZ1STs1U52Ai9TbwcaTWuBE0MVNLrGaRBZN4w7G7QE2y9CNetdsYdDXEHbliuj4HOKsB/5hrgKgXUjLgUQRlplPnxKBxtIwKPEtRswYmZRnC6UtY7MCI0LQYdDQRH+9mfH2jLIgFonuiXmPCc0qJoQMcFM1UKkMyVWsV/9dIH4WlyNNDq3ucebXMC5jkCcMaIJiFmsN1WhFztC8O9j6rMkoDUC5hK4ego/aMhTiV0FYXqZ7qw86Ruk+zj3Dfep1uv9M9wP8m6PQtDFO6zng+tOcnZY0QwA15+5si9KgQiCeovWc56w99yUecqbTP+48rXqR1jnb4kpxzeWgjiJh5zjnKZhArg3ovCV8t3dEGFZFrbjG8MpDgS1hw7GKaco2PXserIdXRipLrUDarU7qpUmIVKlMKdRkHHCivxz3Ax5/Z9MBLwyo6Hg37FNrnnG6mMekDqdnVsiwG2/DCSzXxIxh2J0HJ8Xp/1KRMhQnD4pkn6oICnqupz/WSfg4b1+Ty4NiMZk9DbMx63iEquRlaflxLZ9ugm0S4ZopblURGqe2poHhPR3iNaw3RCj+BEB6UdaE1Np495ZqtAYHYzaiAn4w/dduJJvb71eVeBgWwUMGFZxjP2WtdIHXBapSuid1kMCzk3DDfcw7zSbDvkH/2bciPPx1y3I9+POeYv1k8yY0c64hDVyjvCPrhai3i56wDPfRUd6oaAcQV2KK8vnPBJgdOKWdKxRGEBPER7OBOlCK+LWfFsmdTSBvI0N2VazFML2F4iY6UcKVLlW/5x2ilSVFSwJSMPttNSrZIrLRAXIZBloNSJrpkEppIv6aHBZuYoGcNKX/hm4LZtRb1VO+LKeEw0xab/vS3x1Yz0qaAhdz/y5Praylz+cK0vxDYE5o0Cp8VbD3p8oh2rPBE0xYWNep5o2fqEzxob3xgIrT9nlbOpeLpN4Uh4ayW9oftU806vYNWCtVpkMWp0UK1FbUa53SdQYucDNRtukwX8A7SNRvhZZpsR1/BqUXANR3iJKYgyxrybZcs6Eo/Bg+raYNr/I+qG+kVP4hXt0BMxX3Ke4Jm0CjSg9eD3QYPCg+mjoA4bCTBiVdVysZdDQSYDV/vwFRglT+CU7X+rgLeM6aul7DhFK/mkXeoJ+xpSKugK6MqgmwRFiYULEtBZTu0NOLfDKFOoHTEuk9gqqHDMexcvdSvUEl4JLCBBlFH24zKPZkSCngz0dAStSvMm2mkFj2HnnFyr+Iy0/PENaTmJC7xxGl8yaqTh2RpDzgfh9btLckwLLJoZjYR4YcowCP62eMfR1YZchLkaIcetBQxrzV6vdrb9OMZfvbMZicjSA9QZD6T3R9OK7RKdPG5vSbqUpKrenSsUp936T97sa56tUAeAFxuYTrOb2ipc7ZMlF6RaphyAfbrRj0W/VIkLdPpzDXMp9lspRrXw+vxefOr9Nvq5p7kXrbdzbhDEuwmIUkp102zc1zptfUmyI0l0o2e139cPt1Up+eKq7t10PBJx1PcGWHw90lEaThZDjiVFGl5CKmoxksKwnS7PfW5s5kxqpBSHhXEhNwTXL8+I8LcAYhiGvmfOboD0hLg6Zm71aU2KwPkvhtafwOsriVszLpcufUEuUhkkATxJo8IhO5MS+uawe8vxj9e1Jz16yEHTknr5MBbX2jW4XlH6ncgL/0ZjnzIeeVjVbm9XLekq3RjueYcFz0C/fS7ehvE84tGDrMXmoJ46QqBediujDbEzrYwCb3c0avEL5uSKvbAqX4XCvEdZIjKRvekLPMjpuk2TWdq1MDrTdKAU2Ay2G6Cb0qG0aIeEgcvMo7V8Jxp9wjtlqujK8fKqA4mNeB+kXJgjyIhVBhz8LpRUZotSk2tNp18Gq02UvPJ/EM5fom7UxgdZg6TI0Sqd3nEOSug/pFWOT2zV4Qf8XLeeSTFk43q7DV3EDNFbVyzaO7TQKioYzu+W8aszXjXK2pC2/cH2mN6CWuwnrlGHqdkrh/ciwDJD2KvCCLyF4yHKjXpsg0gMAzWckXBtJCThqpGE3q1FvaTcBFHC6K7/6Bl3zEqLb+Hza+6tI3xYakvL50gUpklUFRYWAuTetsxHtX4nUpKcx2bbbb2vqlQbDtLwth5u8igYVwVCmExS0qge3fo2A3PS7egsrkiu/6iGxm290xehjZ8DdT4cOtGpdhc01v4JXGSkYfqo3VZDNeJ64xUxxR8A3ZGuROgq8bzNzOO5x+1d3kOHNDXQSFevQY7Z5T8ZyBcyws0bVzqxyBbemvKOYRJKlscdGhq9lBo6sAJTfVsaOrt8ORsdH4xVJfnP4xvBpIIJsdUOJ8V4QxiP4PMlIXSvF8QwsLSedbvNaXq/XJ8fX5z/sNwT6rs1LQMbJM6o5rRlLcpSmyEOxq1YbPv9XA6rBA9LtPWW6DTgqIl2kO+GN9QlaaUhra/Fleb/YfiaidxzBUO9TrSSRin92Uc6E+GgECaHbcsXAZc/1gJAtm1NP4Dmx3yZ2g770rbjJRqYXUMS48fZPkVWTTHAFemfb497WRZphqckDpg+yPZYTp85N3J90NvfOW9vhqefD9Q+7Q4IWnkfet/5iFNEL3MimY/M1sPJAot29u1BeXKRY4sCvPIIyy++/ZogoF7GWacvx3bfLPdrvW72/M9s2sBK3gdhgo8bvdxzSgVzQwqNQNpkU7TmFcFtpw9186bk/MRq6b2Ho1xvfqGKFy9nkNerIXjNDM8UNjxj4vGaf+6WiPiLq0UiehJAtLeSo/XeWNn7ca23hrBRRkBJIwOWYZHRx18mkumlxMEyyjncKSkLQnps5rYXQXylTqGBwsndiTia30co49jmPDxIVlMyJW1f0ygJQFVeBbHwsRn/T5YI1yqw2Ytpb8zQy8XD3eZE1PQv1U5XRZnPEJzvb9Qti9LkwU5N+yty0xB9ZLxeXFwSBlG89InpyWnEafuu9K5Ml57NvROxxdvxu8vzmi3CEzMx9ADpJvAaf7YHFhhtlqYp0MKjl2pFM4QwdIb4YLZ39cELE2E5xEXAZ06aKFauZxrIKFE/RVkZpYAD8zTmrIpP6GjMuqU7a4D9yGwvAZA0k+rLJwdaPp5pxm56esK7LIZ455KD6FG9b5y7b84pdquT8cVcHyvpcL2oq309lUKNPa8V/jZd9rO4BigbRP0PLKVmxRPy8gzpR2VX2R2Q7GjX2x572P1hY4qkIuUZWVYYpM4TTYebmNMzzf3zWDrzdE02RaZjyT16loTyuU2DJtARUoYThTSWf+Q35/pKYNufwfukmMlqPSdFALMEWnpf/vfxPxVJay+G/8wvLq45o7c4nnKTZHFLVQguJT7wLuVywoNUj8BVTyi2cKTrWg0vdyD6W0aTUHj15w0BHnLsUjIr9rMcrtk1LgBJkgGZIiO/KTnwKhMEc1RD5quMzpS//d/QSN0gMTlHBCGTNruCCRxwxztcjUWBJU5g6OnhPZvVjtXjXmWZr3EejqLbb7vqgZ+xKcPbENyOVlli6LFnRIr3KoxpiV2JWarSL3WHofaNVwR1Fzf8x1WJMqmzB8ZUfXsHvV07jE2fqlE2nvt9vfNrW2CvA1Vr5Ur+HyghSEqDo6JH+kVsn6Vq0Hhm3M4xuyz4l6LOWg7u9MKPvwtWXgc0ZTtB2VpNZiDTqwBnDQqiaPGlICgtvbFwGBg0zCKFRacfnMYu/kYqq7hfXzsFlSiOZ/bGfBem4SBcVz6tc410zOp1DHvgccoCoBFyLjeSWdGIKAB0H7uvaNoKcBhDClKJHDVyMroFCwVBdWOnQ1jTU2eB1Xu/aw4w8wL1YFjyKw3ZZMqSyLtXuYnWBXRQVkgRqvsTIpurMp7h5Xc1k6mR+O0MoGzYBKCdgiygc5nZSaqJNgwPnqHAvHaXTeKc6apHyvadasuggv2LfRun5evKHikTQm5U6IrL2RXCRZKn7LBx1cYwdhiK9lek9NW9YjOqik35c3IksTsGH1FSzbbfM4EcBmhVPLV7EFg7OTrDW/2lAPNtXVJlJSnWUjdU2fpeRGB6QNtGuyeZVLI5Er+tg7Y6kO4PsLXg4pxy65N8rOqR7VgpdmSva+dZya29TrO5TAzpbJipaNNOq3lxhbKYdIovSIV9RJzWoPXo0aDI1ZdZ8/XgzScqFWVBnpx1O1unYbGy8HxLjofshbVcyJCB81KUBZCuc7Deu2Hrq0fyK4d+Cb8+4nepKPMJp1/SH0M9WwCOE2FJSuK5pbnnlDAyuq7zjYzW6OT18ZrnZvSsFET1TRF9Za4Uge9g4Nn3xw+I3+gGhnbdgDMARR/FK36U4Gqr4SF/kRE6BuiVzpw9ZV4lS1Is1lra6RNoKlMtug8jlMqZ2vtWg/U5H218K5eH7dXHt7Ex17JGVUdRdWcJmBwKFGuLOTlmMlxLXAxj3YGuW6290brGIrOKnqv5KhGaYqJYYq+45Mfx2/eEOWuPbHMib+4Qgn9oPXsnQTODBk+8AsIZRXdpRiiDawZcj+eXL3z3l86gkHlUvAPO4DKlPS1B4Bdnby7hAsnrdxvhQBnoecGPSjyRXZBb/sBRaBVChrJ0V+me8O/XY7GVyc3Y4K8CRRuLMdNxMEE0DWc1eujdh1iu/fI2VV9Pc0isrrdbqd7CC+bYsOi4l0qLHJ68xCdrNk4e940Z3cyCuVzvngu2DPx2N2KCFlNQyJB+l/y0fZUTHU5Ojkdvh2PzoZXvtBt79HM1QM7PJvbYQ0T3KHbtKe67K5PmnrgXGj/nSoyylfqdxoY3+cwMbXz3aNjKXegYkWq0aMRNnfV4l+P31+dYjr0KWdkm7td3yG/2pTR/1/NDs4VlS7Btsbhr4Q5KBOuj9WBAS08Eq9yxrkjIlp8/KQzgXRL580sbXbgf5U5Au3KalC2KjRx0john4OmJg0ny8wre0EOluaJ3Bn8MdUtyrKbI/++ttySANlectU97HSP9FloLnu8fLm9CO3KOgt82J7V7U245sYXHeCkG7TjQR9NUT4xUC80dKi8B2SbAFt9pg0IK3sIjZQlYJb3iZZ83Zdzt1jLV8hWT6p4gPaX4EtJj4kIPT7Kg7s6j9M0k0M0DizFR3LEjFlaRuSE1cjZ3V+md1Aw+xQ+1JvoLW/JAAToxfrUX8br3H3bVL8S96A+8i1rdsRkGzJlh5OAsMDBzgovwhHL9VLxQ2BGmKC86Wx01b5veSav8TmZ2zWX292LmrMb+sCLuWaKl/xcizIJQe7OuUA/PtuyWWeGcisKz/JTrnAvN704N3nvS++FFCzsVHnSn6QiT3L4natp3RVhQXJO0XTJDf64qzw9v9PumWUnr8GRtrsJiLI3fNFsRWn+wSjNwv1O6WDeYVEnX9kW9AD9Cl9Uiutp2c2ut/LszEZlE6DeLTDlGum1VJUFBmyN+rxt0OhbIeVYAZ92ET6058nuXHvyDfv5nKC+E/Exdl+DFHf72cAcI2QHZraBlUcESG/KkdenYveGOLsrXMLX0/VSsOVmtiN/WVa4mL46G/ZmTUgFOYz/bHeZRlKaywcyc/EwVR3w6vkSWroLW7xxzedNrb4cgNMCTvGpLI19kJYCDk/Q+SksK+AuFWV8qB2XJmMxWyXKgimPX6okzBzW0DOgD9VozGbp/CVX8zEi4sZzduV2VRXQIzIFOs32UjaHuAtka0xt2EoOwLjgC6K0puEkLbmBp8Q5jkN2fNstGGzAgYip6NQGx+UdOSrI8C5f8znk79P5nWluQuhUsb8Iv7I3o1Pu1daB7nIXd+UQr9oum/+sh/n1Uwl2H5xQWZJ1wsrS0XdeHsxDXUqtGnysEhcDbpaURNqU+fUR+xP6MMiar/Ggm3H1h8jeBfPszmhfysHgxxxIr0J5gtd06goDeSDsIfUfgu1RMaVnirO3yiZlk5IOuxPbjN/fnI6xfpUSL49zs1kmVb0N+EiZjvs4tcNN8p8oxC6ZYjzbKkOjJ6PxxXfX52eynS8JiPvGo2vKJV/KViU+fwZPnv4gGO42hPaFmEhpqXfFZWdlPOH0nGvotTBAKuz8cQJaqgB0h9Tw5Gr0kyt4dKSzU61E25R1QFOi+HQkigAXmhqgG48UkP2jDHt8bkbNYWBn6WGHgW7LGZeXOqhcTpfONBje2HbbrKs3UDqbLFyhA26jo8Z2CpGSueN+NWLNiUPHHeUEYptytPRsWaArgTUda6cN2+hvxzlKyelZjeDB0TNG89UE52iXOtyZyt3KWY77u3ZKO1lM20HT1uHXWjLpk/KtB1raaqeMK8nfrxjUEv16YUoXvfaXP0QGH3jZ1ks5Cg1+nPzJEvMXNh6gwK4oP/DQ2xU1+7XOOw9W/44I67Wtd+3Zrab75oLuidMFXY2gLbrZAFgaeHvwDiOjsoDslhVU+f1ef+cDl2152Qf36B69FZs2O1Cga8fBoUb5unvu7Sm1xhOgEZORtYc7W+uyxV/VM6XLP94inSSLTpk7+cMIlpxqXJAnTUJH5SZUACsFho2z3kHnRP44gkxbuEOZG8AZ3dnqNavMraL3raLnfsoDFT1f6nIKI/Q610P+SyG/9DVtq6R8bQB4gOVbvrt7RPFhNYFf2oqXckEUfkddlfRqe0wBcylYLofnVyzC8Q5TYHeQ3vlz9snVkXyrFs7INbL/dIQ5+n1oW61YEsVpEmNmhLR9wG4Z4nhkebm2t0gjc7FHA30eoGazLdv0sGUyUVChUyongZfyjFO84Voxc85pkptdNT//PDqkCpcPLfUz1eKMDz9IwW5p7UgpdtwEdm5oaMcdavPeOE8a/3EujNK/ye6dVMwGNiQg7trMu6iEFMucIZU+zejU2ns54v82TajYhbaplmDrap3wWRUMozL5wn/RZz39OJswbAz4iE+JZcvfI5GrEiyZRIVXTSWgMXHm6C+68B/R0iFwUj+29s7oOnQlThcLvQ+T/qAOvpaaUF9gpfn/AWLhn08EbAAA"
}
}


In [ ]:
# ===================== Cell 2 — materialise embedded sources + import =====================
import io
import os
import gzip
import base64
import hashlib
import json
import shutil
import contextlib
import importlib.util
import logging
import pathlib
import sys

import yaml

WORK = pathlib.Path(WORK_DIR).expanduser().resolve()
SRC = WORK / "src"
(WORK / "outputs").mkdir(parents=True, exist_ok=True)
(WORK / "data" / "interim").mkdir(parents=True, exist_ok=True)
SRC.mkdir(parents=True, exist_ok=True)

# ---- write embedded sources (sha256-verified) ----
print("Embedded-source manifest (name · bytes · sha256):")
for name, rec in _EMBEDDED.items():
    raw = gzip.decompress(base64.b64decode(rec["gz_b64"]))
    digest = hashlib.sha256(raw).hexdigest()
    assert digest == rec["sha256"], f"integrity mismatch for {name}"
    assert len(raw) == rec["n_bytes"], f"size mismatch for {name}"
    dest = SRC / name if name.endswith(".py") else (WORK / name)
    dest.write_bytes(raw)
    print(f"  {name:26s} {rec['n_bytes']:7d}  {digest}")
    print(f"    -> {dest}")

# ---- data/raw -> DATA_DIR symlink (01_audit stats REPO_ROOT/data/raw/<combine_tracking>) ----
link = WORK / "data" / "raw"
if link.is_symlink():
    link.unlink()
elif link.exists():
    shutil.rmtree(link)
os.symlink(os.path.abspath(DATA_DIR), link)

# ---- patch the embedded config: absolute runtime paths; game side out of scope ----
CFG = yaml.safe_load((WORK / "config.yaml").read_text(encoding="utf-8"))
CFG["paths"]["raw"] = os.path.abspath(DATA_DIR)
CFG["paths"]["interim"] = str(WORK / "data" / "interim")
CFG["paths"]["outputs"] = str(WORK / "outputs")
CFG["logging"]["run_log"] = str(WORK / "outputs" / "run_log.csv")
CFG["mode"]["full"]["run_game_side"] = False   # Phase 4/5 game side is out of scope this cycle
(WORK / "config.yaml").write_text(yaml.safe_dump(CFG, sort_keys=False), encoding="utf-8")
print("\nPatched config ->", WORK / "config.yaml")
print("  paths.raw      =", CFG["paths"]["raw"])
print("  paths.interim  =", CFG["paths"]["interim"])
print("  paths.outputs  =", CFG["paths"]["outputs"])
print("  logging.run_log=", CFG["logging"]["run_log"])
print("  mode.full.run_game_side =", CFG["mode"]["full"]["run_game_side"])

# ---- import each stage module in-process (REPO_ROOT resolves to WORK) ----
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

def _load(mod_name, filename):
    spec = importlib.util.spec_from_file_location(mod_name, SRC / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

S_AUDIT = _load("bdb27_audit", "01_audit.py")
S_FEATURES = _load("bdb27_features", "02_features.py")
S_MODEL = _load("bdb27_model", "03_combine_model.py")
S_PANEL = _load("bdb27_panel", "03b_metric_panel.py")
S_WARMUP = _load("bdb27_warmup", "03d_warmup_phenotype.py")
print("\nmodules imported: 01_audit, 02_features, 03_combine_model, 03b_metric_panel, 03d_warmup_phenotype")


def run_stage(label, fn, mode=MODE, force=True):
    """Run one stage in-process; capture its stdout+stderr and return code."""
    lg = logging.getLogger("bdb27")
    saved_handlers, saved_prop = lg.handlers[:], lg.propagate
    lg.handlers, lg.propagate = [], False
    out, err = io.StringIO(), io.StringIO()
    try:
        with contextlib.redirect_stdout(out), contextlib.redirect_stderr(err):
            rc = fn(mode, force)
    finally:
        lg.handlers, lg.propagate = saved_handlers, saved_prop
    text = out.getvalue() + err.getvalue()
    print(f"[{label}] run(mode={mode!r}, force={force}) -> return code {rc}")
    tail = "\n".join([ln for ln in text.strip().splitlines() if ln.strip()][-10:])
    if tail:
        print("--- captured stage log (tail) ---")
        print(tail)
    return rc


## Stage 1 — Phase 1 audit (go/no-go gate)

Runs `01_audit.run(MODE, force=True)` in-process. Produces `outputs/audit/*` and the gate verdict.
Audit questions: (a) timing present, (b) attempts/position, (c) drill order varies, (d) distance
reliability, (e) matched Combine→NFL (**PENDING FULL RUN** — game side out of scope), (f) missingness/units,
(g) numbering gaps, (h) position scope.

In [ ]:
rc = run_stage("01_audit", S_AUDIT.run)
assert rc == 0, ("01_audit exit=%s (gate not GO)" % rc)


In [ ]:
# ---- render: audit gate (table 1) ----
import json
import pandas as pd

A = WORK / "outputs" / "audit"
audit = json.loads((A / "audit_summary.json").read_text(encoding="utf-8"))
a, c, d, rec, gate = audit["a"], audit["c"], audit["d"], audit["recommendation"], audit["gate"]
emin = audit["e"][0]
rows = [
    ("provenance", audit.get("provenance")),
    ("(a) timing field present", a["timing_field_present"]),
    ("(a) attempt~time agreement", f'{a["order_agree_frac"]} (min {a["order_violations"]} violation(s))'),
    ("(c) drill order varies across players", c["drill_order_varies_across_players"]),
    ("(c) max pairwise flip share", f'{c["max_pairwise_flip_share"]}  (thr {c["order_flip_threshold"]})'),
    ("(d) distance (provided `dis`) reliable", d["reliable"]),
    ("(d) share of attempts >5% off (weighted)", d["share_over_tol_weighted"]),
    ("(g) overall numbering-gap share", gate["overall_gap_share"]),
    ("(h) study population (locked, D13)", rec["position_group"]),
    ("(h) players reaching >=3 attempts", f'{rec["players_ge3_attempts"]}/{rec["players_total"]}'),
    ("(e) matched Combine->NFL per group", emin["status"]),
    ("gate verdict", gate["verdict"]),
]
print(pd.DataFrame(rows, columns=["audit check", "value"]).to_string(index=False))
print("\n(e) is PENDING FULL RUN — the game-side match needs the full Kaggle game-tracking files.")


## Stage 2 — Phase 2 Combine features

Runs `02_features.run(MODE, force=True)`. Computes per-attempt kinematics from x/y/time, the
observed+imputed cumulative prior load (imputed rows carry **no** performance), within-(drill_type, position)
standardisation and the VIF check. Outputs `outputs/features/*`; the DB-only study handoff is
`combine_features_study.parquet`.

In [ ]:
rc = run_stage("02_features", S_FEATURES.run)
assert rc == 0

In [ ]:
# ---- render: Phase-2 features summary (table 2) ----
F = WORK / "outputs" / "features"
fd = pd.read_csv(F / "feature_diagnostics.csv")
prov2 = fd["provenance"].iloc[0]
kv = dict(zip(fd["metric"], fd["value"]))
vif_flag = float(kv["vif_prior_load_yd"]) > float(kv["vif_flag_threshold"])
rows = [
    ("provenance", prov2),
    ("attempt rows observed / imputed", f'{kv["rows_observed"]} / {kv["rows_imputed"]}'),
    ("study population", kv["study_population"]),
    ("study observed rows / players", f'{kv["rows_study_observed"]} / {kv["n_players_study"]}'),
    ("n drill types", kv["n_drill_types"]),
    ("standardise level", CFG["features"]["standardize_level"]),
    ("min std-group n (config)", CFG["features"]["min_std_group_n"]),
    ("VIF(prior_load_yd, first_attempt)", f'{float(kv["vif_prior_load_yd"]):.4f} (flag thr {kv["vif_flag_threshold"]}; exceeds={vif_flag})'),
    ("imputed share of observed", kv["imputed_share_of_observed"]),
    ("rest_s policy", kv["rest_use"]),
]
print(pd.DataFrame(rows, columns=["Phase-2 feature", "value"]).to_string(index=False))


## Stage 3 — Phase 3 Combine fatigue model (D20)

Runs `03_combine_model.run(MODE, force=True)`. Pre-registered spec: `perf_z ~ load + load² + first_attempt
+ C(drill) + (1 + load | player)` with the documented convergence fallback chain, model-based slope
reliability and a ≥1000-shuffle within-player permutation test. **Stop rule:** reliability < 0.2 ⇒ report
the **NULL** as the finding.

In [ ]:
rc = run_stage("03_combine_model", S_MODEL.run)
assert rc == 0

In [ ]:
# ---- render: Phase-3 D20 headline (table 3) + FIGURE 1 ----
%matplotlib inline
import json
import matplotlib.pyplot as plt
import duckdb
import numpy as np
import pandas as pd

M = WORK / "outputs" / "model"
ms = json.loads((M / "combine_model_summary.json").read_text(encoding="utf-8"))
reliab = float(ms["reliability"]); thr = float(ms["reliability_stop_threshold"])
rows = [
    ("estimator chosen", ms["estimator_chosen"]),
    ("slope reliability", f'{reliab}  (stop threshold {thr})'),
    ("stop rule triggered", ms["stop_rule_triggered"]),
    ("population load coef (per yd)", ms["population_load_coef"]),
    ("population load p", ms["population_load_p"]),
    ("permutation p (meta_slope)", ms["permutation"]["p"]),
    ("baseline late−early mean", ms["baseline"]["mean"]),
    ("model obs / players", f'{ms["n_obs"]} / {ms["n_players"]}'),
    ("FINDING", "NULL" + (" (reliability < stop threshold)" if reliab < thr else "")),
]
print(pd.DataFrame(rows, columns=["Phase-3 D20", "value"]).to_string(index=False))

con = duckdb.connect()
sl = con.execute(
    f"SELECT slope_raw, slope_shrunk, reliability FROM read_parquet('{(M / 'combine_player_slopes.parquet')}')"
).fetchdf()
var_shrunk = float(np.var(sl["slope_shrunk"], ddof=1))
print(f'\nper-player slopes: n={len(sl)}, Var(slope_raw)={np.var(sl["slope_raw"], ddof=1):.3e}, '
      f'Var(slope_shrunk)={var_shrunk:.3e}  -> ~zero between-player slope variance')

fig, ax = plt.subplots(figsize=(6.6, 3.6))
ax.hist(sl["slope_shrunk"], bins=30, color="#4C72B0", edgecolor="white")
ax.axvline(0.0, color="black", lw=0.8)
ax.axvline(float(ms["population_load_coef"]), color="crimson", lw=1.4, ls="--",
           label=f'population coef = {float(ms["population_load_coef"]):.3e}')
ax.set_xlabel("EB-shrunk per-player fatigue slope (per yard prior load)")
ax.set_ylabel("players")
ax.set_title(f"FIGURE 1 — per-player Combine fatigue slopes\n"
             f"between-player variance ~ 0 (reliability {reliab}) → NULL")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## Stage 3b — Phase 3b metric panel (D21, pre-registered)

Runs `03b_metric_panel.run(MODE, force=True)`. Tests whether a *better* load metric rescues per-player slope
reliability ≥ 0.2 across the full 7×3 = 21 (load × outcome) cell panel, using the same fallback chain,
reliability formula and stop rule as Phase 3.

In [ ]:
rc = run_stage("03b_metric_panel", S_PANEL.run)
assert rc == 0

In [ ]:
# ---- render: Phase-3b gate cells (table 4) + FIGURE 2 ----
pjson = json.loads((M / "metric_panel.json").read_text(encoding="utf-8"))
cells = pd.DataFrame(pjson["cells"]).sort_values(["load_code", "outcome_code"]).reset_index(drop=True)
cells["cell"] = cells["load_code"] + "x" + cells["outcome_code"]
met = cells[cells["reliability_met"]]
print(f'panel cells: {len(cells)};  reliability >= {CFG["panel"]["reliability_stop_threshold"]}: {len(met)} '
      f'(metric-robust null — the fatigue direction does not survive better load metrics)')
print(met[["cell", "load_metric", "outcome_metric", "estimator", "reliability", "load_coef", "load_p"]]
      .to_string(index=False))

thr_p = float(CFG["panel"]["reliability_stop_threshold"])
colors = ["#C44E52" if m else "#9E9E9E" for m in cells["reliability_met"]]
fig, ax = plt.subplots(figsize=(8.2, 3.6))
ax.bar(cells["cell"], cells["reliability"], color=colors)
ax.axhline(thr_p, color="black", lw=1.0, ls="--", label=f"gate = {thr_p}")
ax.set_ylabel("slope reliability")
ax.set_title("FIGURE 2 — reliability by (load x outcome) cell; red = meets the >=0.2 gate")
ax.tick_params(axis="x", labelrotation=45, labelsize=8)
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## Stage 3d — Phase 3d warm-up phenotype (D23/D24)

Runs `03d_warmup_phenotype.run(MODE, force=True)`. **Headline pivot:** the only reliable individual combine
signal (D21) is a *positive* load coefficient on explosive output — i.e. warm-up / post-activation
potentiation, not fatigue. The make-or-break step separates "warmed up" from "ran this drill later in the
fixed order" (de-confounding). D24 sets the **governing strict reading** of the D23 PASS/FAIL rule.

In [ ]:
rc = run_stage("03d_warmup_phenotype", S_WARMUP.run)
assert rc == 0

In [ ]:
# ---- render: Phase-3d de-confounding (table 5) + FIGURE 3 ----
wj = json.loads((M / "warmup_summary.json").read_text(encoding="utf-8"))
head = wj["headline"]
dc = pd.read_csv(M / "warmup_deconfounding.csv")
dc["cell"] = dc["load_code"] + "x" + dc["outcome_code"]
print(f'governing reading (D24): {head["deconf_pass_governing_reading"]}  ->  deconf_pass = {head["deconf_pass"]}')
print(f'primary-cell deconf_pass = {head["deconf_pass_primary"]};  '
      f'all-accel deconf_pass = {head["deconf_pass_all_accel"]}  =>  warm-up NOT CLAIMED (exploratory only)')
print(dc[["cell", "load_metric", "outcome_metric", "raw_coef", "adj_coef", "adj_retain_frac", "deconf_pass"]]
      .to_string(index=False))
bc = dc["within_contrast_battery"]
print(f'\nsurviving signal: positive within-drill rep1->rep2 battery contrast z = '
      f'+{bc.min():.4f} .. +{bc.max():.4f} (n={int(dc["within_contrast_battery_n"].iloc[0])} players) — exploratory')
print(f'D20/D21 fatigue null remains the headline; warm-up median reps-to-peak = '
      f'{head["median_reps_to_peak"]:.3f} (identified {head["median_reps_to_peak_identified"]:.3f}, '
      f'frac censored {head["frac_censored_at_grid_max"]:.3f})')

thr_r = float(CFG["warmup"]["deconf_retain_frac"])
colors = ["#55A868" if p else "#C44E52" for p in dc["deconf_pass"]]
fig, ax = plt.subplots(figsize=(8.2, 3.6))
ax.bar(dc["cell"], dc["adj_retain_frac"], color=colors)
ax.axhline(thr_r, color="black", lw=1.0, ls="--", label=f"retain gate = {thr_r}")
ax.set_ylabel("adjusted coef retention |adj|/|raw|")
ax.set_title("FIGURE 3 — de-confounding retention per cell; green = PASS, red = FAIL (D23/D24)")
ax.tick_params(axis="x", labelrotation=45, labelsize=8)
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()


## Phases 4 & 5 — NOT implemented this cycle

> **NOT implemented — requires the full Kaggle game tracking.**

- **Phase 4 (`00_thresholds` + `04_game_features`):** per matched player, model high-intensity effort
  intensity vs. their own early-game baseline against cumulative in-game distance, snaps-in-drive and
  time-since-rest, with the minimum-snaps rule; extract per-player decay slopes and the same model-based
  reliability/stop rule.
- **Phase 5 (`05_link`):** regress game decay slope on the Combine fatigue slope, controlling for baseline
  speed and size; account for error in both slopes; player-grouped CV with held-out ΔR²; placebo and
  sensitivity tests.

Neither is runnable here: the game-tracking files are not in this dataset slice, and the pre-registered
**stop rule has already fired on the combine side** (Phase-3 reliability 0.0 < 0.2), so the Phase-5 link is
declared out of scope for this cycle. No placeholders or synthetic game results are produced.
See `notes/pending_full_run.md` and `notes/limitations.md`.


## Summary

**Combine-side result (reproduced from HEAD `f45e3af`), all numbers read from the produced artefacts:**

- **Phase 1 audit — GO.** Timing fields present (attempt~time agreement ≈ 0.999); drill order varies
  (only the shuttle/3-cone relative order flips); provided `dis` is **unreliable** vs the x/y path;
  ≈32% of player-drill groups have numbering gaps; **DB-only** study population (122/122 players reach
  ≥3 attempts). (e) matched Combine→NFL is **PENDING FULL RUN**.
- **Phase 2 features.** 1432 study (DB) observed attempt rows / 122 players; load imputed only (imputed rows
  carry no performance); VIF well below threshold. Confirms the Phase-3 handoff.
- **Phase 3 (D20) — NULL.** Estimator `per_player_eb`; between-player slope variance ≈ 0 ⇒ reliability
  **0.0 < 0.2** ⇒ stop rule fires ⇒ **the finding is a NULL**. The pooled population load coefficient is
  small (p≈0.0055, permutation p≈0.03), but with zero cross-player slope variance those slopes cannot
  support the Phase-5 link.
- **Phase 3b (D21) — metric-robust.** Of 21 (load × outcome) cells, only the 3 accel-outcome cells meet the
  0.2 gate — a *positive* load effect on explosive output (warm-up direction), not fatigue. The fatigue
  direction remains a metric-robust null.
- **Phase 3d (D23/D24) — exploratory.** Under the **governing strict reading** (`strict_all_accel`)
  de-confounding **fails** ⇒ warm-up is **NOT claimed**; the surviving signal is a small positive
  within-drill rep1→rep2 battery contrast (exploratory). The D20/D21 fatigue null remains the headline.

**Provenance.** All combine-side outputs are stamped with the run mode's provenance label. Because the
combine input data are **complete**, the combine numbers are **REAL** in both modes (`full` ⇒ `FULL`).

**Caps & fallback.** See `notes/rules.md` for the official limits (≤2,000 words; <10 tables/figures) and the
stricter internal ≤5-figure cap. A CLI fallback that runs the identical per-stage scripts
(`python src/01_audit.py --mode full`, …) is documented in `README_RUN.md`.

**Regeneration.** This notebook is generated deterministically by `tools/build_notebook.py`
(`python tools/build_notebook.py`), which embeds `src/*.py` + `config.yaml` as sha256-verified
gzip+base64 payloads.
